In [0]:
%pip install "numpy<2.0" --quiet
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 08 — Anomaly Detection & LLM Validation (v4 — IDP-Native + Web-Augmented)
# MAGIC
# MAGIC **Primary source:** `gold_idp_enriched` (~955 rows, 190 cols — no merge needed, all IDP fields present)
# MAGIC **Regional context:** `gold_regional_summary` (17 regions)
# MAGIC
# MAGIC ### Architecture
# MAGIC
# MAGIC | Layer | Method | Scope |
# MAGIC |-------|--------|-------|
# MAGIC | 0  | Data integrity repair (negatives, type coercions) | All rows |
# MAGIC | 1a | Evidence-absence confidence (IDP-native: `evidence_weight`, `source_trust`) | All rows |
# MAGIC | 1b | Duplicate identity detection (name+region hash) | All rows |
# MAGIC | 1c | Statistical anomaly detection — evidence-weighted, IDP-native | All rows |
# MAGIC | 1d | Capability graph reasoning (`dependency_gaps`, `service_richness`, `infrastructure_completeness`) | All rows |
# MAGIC | 1e | Peer-group comparative scoring (same region + facility type) | All rows |
# MAGIC | 1f | Service maturity tiers (IDP confidence-aware) | All rows |
# MAGIC | 1g | Continuity & fragility risk | All rows |
# MAGIC | 1h | Composite scoring + data poverty classification | All rows |
# MAGIC | 2  | LLM validation + optional web search enrichment (priority batch) | Top-N |
# MAGIC | 3  | Regional prioritization engine | All regions |
# MAGIC
# MAGIC **Outputs:**
# MAGIC - `virtue_foundation.ghana.gold_anomaly_flags`
# MAGIC - `virtue_foundation.ghana.gold_anomaly_report`
# MAGIC - `virtue_foundation.ghana.gold_regional_priority`

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json
import re
import time
import warnings
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

import mlflow
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.getOrCreate()
print(f"Run at: {datetime.now(timezone.utc).isoformat()}")


# COMMAND ----------

class AnomalyConfig:
    # ── Source tables ──────────────────────────────────────────────────────────
    IDP_TABLE      = "virtue_foundation.ghana.gold_idp_enriched"   # PRIMARY (IDP-enriched, ~955 rows)
    REGIONAL_TABLE = "virtue_foundation.ghana.gold_regional_summary"

    # ── Output tables ──────────────────────────────────────────────────────────
    ANOMALY_TABLE  = "virtue_foundation.ghana.gold_anomaly_flags"
    REPORT_TABLE   = "virtue_foundation.ghana.gold_anomaly_report"
    PRIORITY_TABLE = "virtue_foundation.ghana.gold_regional_priority"

    # ── Databricks / LLM ──────────────────────────────────────────────────────
    DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl", "").rstrip("/")
    try:
        DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception as _e:
        raise RuntimeError(f"❌ Failed to retrieve authentication token: {_e}")

    LLM_MODEL    = "databricks-meta-llama-3-3-70b-instruct"
    LLM_ENDPOINT = (
        f"https://{spark.conf.get('spark.databricks.workspaceUrl','').rstrip('/')}"
        f"/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"
    )
    MLFLOW_EXP = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"

    # ── Anomaly thresholds (calibrated to IDP distribution) ───────────────────
    # Capability inflation: IDP cap_cnt / equip_cnt ratio
    CAPABILITY_INFLATION_RATIO   = 4.0    # >= 4x ratio required (relaxed from 3.5)
    CAPABILITY_INFLATION_MIN_CAP = 4      # min capability_count to flag

    # Ghost facility (multi-signal: completeness + ghost_prob + operational signals)
    GHOST_COMPLETENESS_CUTOFF    = 0.40
    GHOST_MIN_CAPABILITY_CLAIMS  = 3
    GHOST_PROB_MIN               = 0.55   # ghost_probability_score must agree

    # Evidence confidence thresholds
    HIGH_COMPLETENESS_THRESHOLD  = 0.65
    LOW_COMPLETENESS_THRESHOLD   = 0.38

    # Data poverty: low completeness + low ghost = missing data, NOT anomaly
    DATA_POVERTY_COMPLETENESS_MAX = 0.32
    DATA_POVERTY_GHOST_MAX        = 0.40

    # Procedure breadth
    PROC_EQUIP_RATIO_THRESHOLD    = 4.5

    # ICU infrastructure
    ICU_MIN_EQUIPMENT             = 2
    ICU_MIN_CAPABILITY            = 3

    # Doctor:bed ratio plausibility
    MAX_DOCTOR_BED_RATIO          = 1.5

    # Peer comparison
    PEER_ZSCORE_THRESHOLD         = 2.5

    # IDP-native: infrastructure vs maturity mismatch
    MATURITY_INFRA_MISMATCH_MAT   = 0.50   # healthcare_maturity_score
    MATURITY_INFRA_MISMATCH_INFRA = 0.15   # infrastructure_completeness_score

    # IDP-native: service richness vs equipment mismatch
    RICHNESS_EQUIP_MIN_RICHNESS   = 0.55
    RICHNESS_EQUIP_MAX_EQUIP      = 0      # equipment_count == 0

    # Composite risk score weights
    W_FLAG_NORM      = 0.30
    W_GHOST_PROB     = 0.28   # ghost_probability_score — primary IDP signal
    W_QUALITY_RISK   = 0.18   # quality_risk_score from IDP
    W_CALIBRATED_ABS = 0.14   # evidence_absence calibrated by completeness
    W_CONTINUITY     = 0.10

    # Risk thresholds
    RISK_CRITICAL_COMPOSITE = 0.58
    RISK_CRITICAL_FLAGS_MIN = 3
    RISK_CRITICAL_COMP_MIN  = 0.50
    RISK_HIGH_COMPOSITE     = 0.38
    RISK_MEDIUM_COMPOSITE   = 0.22

    # LLM batch
    MAX_LLM_BATCH    = 50
    LLM_MIN_SCORE    = 0.28

    # Web search
    WEB_SEARCH_ENABLED = True
    WEB_SEARCH_TIMEOUT = 6


cfg = AnomalyConfig()
print(f"LLM endpoint : {cfg.LLM_ENDPOINT}")
print(f"Source table : {cfg.IDP_TABLE}")


# COMMAND ----------
# MAGIC %md ## 1. Utility Functions

# COMMAND ----------

def ensure_list(x) -> List[str]:
    """Safely coerce any value to a clean list of non-empty strings.
    Uses explicit len() check to avoid numpy DeprecationWarning."""
    if x is None:
        return []
    # Handle numpy / pandas arrays explicitly
    if hasattr(x, "__len__") and not isinstance(x, (str, list, tuple, dict)):
        try:
            if len(x) == 0:
                return []
            return [str(v) for v in x
                    if v is not None and str(v).strip().lower() not in ("", "null", "nan", "none")]
        except Exception:
            pass
    if isinstance(x, float) and x != x:  # NaN
        return []
    if isinstance(x, (list, tuple, set)):
        return [str(v) for v in x
                if str(v).strip().lower() not in ("", "null", "nan", "none")]
    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None", "none"):
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v) for v in parsed
                        if str(v).strip().lower() not in ("", "null", "nan", "none")]
            return [str(parsed)]
        except Exception:
            if "|" in s:
                return [v.strip() for v in s.split("|") if v.strip()]
            return [s]
    return [str(x)]


def safe_float(v, d: float = 0.0) -> float:
    try:
        if v is None or str(v).strip() in ("nan", "None", "null", ""):
            return d
        f = float(v)
        return d if (f != f) else f  # NaN check
    except Exception:
        return d


def safe_int(v, d: int = 0) -> int:
    try:
        if v is None or str(v).strip() in ("nan", "None", "null", ""):
            return d
        return int(float(v))
    except Exception:
        return d


def safe_str(v, d: str = "") -> str:
    if v is None:
        return d
    s = str(v).strip()
    return d if s in ("None", "nan", "null", "", "NaN") else s


def safe_bool(v, d: bool = False) -> bool:
    if v is None:
        return d
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    return str(v).strip().lower() in ("true", "1", "yes")


def coerce_bool_col(series: pd.Series) -> pd.Series:
    return series.apply(safe_bool).astype(bool)


def parse_capability_graph_summary(v) -> Dict:
    """Parse capability_graph_summary JSON string safely."""
    if not v or pd.isna(v) if not isinstance(v, str) else False:
        return {}
    try:
        s = str(v).strip()
        return json.loads(s) if s else {}
    except Exception:
        return {}


# ── LLM call with retry ────────────────────────────────────────────────────────
def call_llama(
    messages:      List[Dict],
    system_prompt: Optional[str] = None,
    max_tokens:    int   = 1200,
    temperature:   float = 0.15,
    retries:       int   = 3,
) -> str:
    full = []
    if system_prompt:
        full.append({"role": "system", "content": system_prompt})
    full.extend(messages)
    headers = {
        "Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}",
        "Content-Type":  "application/json",
    }
    payload = {
        "messages":    full,
        "max_tokens":  max_tokens,
        "temperature": temperature,
        "top_p":       0.92,
        "stream":      False,
    }
    for attempt in range(retries):
        try:
            r = requests.post(cfg.LLM_ENDPOINT, headers=headers, json=payload, timeout=60)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.HTTPError as e:
            code = getattr(e.response, "status_code", None)
            if code == 429:
                wait = 2 ** attempt * 10
                print(f"  [rate-limit] waiting {wait}s...")
                time.sleep(wait)
            elif attempt == retries - 1:
                return f"[LLM error: {str(e)[:120]}]"
            else:
                time.sleep(5 * (attempt + 1))
        except Exception as e:
            if attempt == retries - 1:
                return f"[LLM error: {str(e)[:120]}]"
            time.sleep(5 * (attempt + 1))
    return ""


def parse_json_llm(raw: str) -> Dict:
    """Robustly extract JSON from LLM response."""
    if not raw:
        return {}
    clean = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```$", "", clean).strip()
    s, e  = clean.find("{"), clean.rfind("}")
    if s != -1 and e != -1:
        clean = clean[s: e + 1]
    try:
        return json.loads(clean)
    except Exception:
        clean = re.sub(r",\s*}", "}", clean)
        clean = re.sub(r",\s*]", "]", clean)
        try:
            return json.loads(clean)
        except Exception:
            return {}


# ── Web search utilities ───────────────────────────────────────────────────────
def web_search_duckduckgo(query: str, timeout: int = cfg.WEB_SEARCH_TIMEOUT) -> str:
    """
    Query DuckDuckGo Instant Answer API (no API key required).
    Returns a concise text snippet or empty string on failure.
    """
    try:
        q_enc  = requests.utils.quote(query)
        url    = f"https://api.duckduckgo.com/?q={q_enc}&format=json&no_html=1&skip_disambig=1"
        resp   = requests.get(url, timeout=timeout, headers={"User-Agent": "VirtueFoundationAgent/1.0"})
        if resp.status_code != 200:
            return ""
        data     = resp.json()
        abstract = safe_str(data.get("Abstract", ""))[:400]
        related  = " | ".join(
            safe_str(r.get("Text", ""))[:120]
            for r in data.get("RelatedTopics", [])[:4]
            if isinstance(r, dict) and r.get("Text")
        )
        result = f"{abstract} {related}".strip()
        return result[:600]
    except Exception:
        return ""


def web_search_facility(name: str, region: str, facility_type: str) -> str:
    """
    Search for facility-specific web information to augment LLM validation.
    Returns text snippet for injection into LLM context.
    """
    if not cfg.WEB_SEARCH_ENABLED:
        return ""
    queries = [
        f"{name} Ghana hospital services",
        f"{name} {region} Ghana healthcare",
        f"{name} Ghana medical facility",
    ]
    for q in queries:
        result = web_search_duckduckgo(q)
        if result and len(result) > 50:
            return f"[WEB: {result[:500]}]"
    return ""


def llm_knowledge_search(facility_name: str, region: str, anomaly_context: str) -> str:
    """
    Use LLM's training knowledge to validate anomaly flags for known Ghana facilities.
    This leverages the LLM as a knowledge source about healthcare facilities.
    """
    prompt = (
        f"You are a Ghana healthcare expert. A facility named '{facility_name}' in "
        f"{region}, Ghana has been flagged with potential anomalies:\n{anomaly_context[:400]}\n\n"
        f"Using your knowledge of Ghana healthcare:\n"
        f"1. Is this a known facility? What do you know about it?\n"
        f"2. Are the anomaly flags plausible for this type of facility?\n"
        f"3. Are there any known special circumstances (e.g., teaching hospital, referral centre)?\n\n"
        f"Return JSON only:\n"
        f'{{"is_known_facility": true|false, "facility_profile": "<brief>", '
        f'"flag_plausibility": "LIKELY_VALID|LIKELY_FALSE_POSITIVE|UNCERTAIN", '
        f'"special_context": "<if any>", "confidence": 0.0-1.0}}'
    )
    raw    = call_llama([{"role": "user", "content": prompt}],
                        max_tokens=350, temperature=0.05)
    return parse_json_llm(raw)


# COMMAND ----------
# MAGIC %md ## 2. Load Source Tables

# COMMAND ----------

print("Loading gold_idp_enriched (primary IDP-enriched source)...")
try:
    idp_sdf = spark.table(cfg.IDP_TABLE)
    idp_df  = idp_sdf.toPandas()
    print(f"  Loaded: {len(idp_df):,} rows × {len(idp_df.columns)} cols")
except Exception as e:
    raise RuntimeError(f"❌ Cannot load IDP table: {e}")

print("Loading gold_regional_summary (regional context)...")
try:
    reg_df = spark.table(cfg.REGIONAL_TABLE).toPandas()
    print(f"  Loaded: {len(reg_df):,} rows × {len(reg_df.columns)} cols")
except Exception as e:
    print(f"  Regional table unavailable: {e}. Proceeding without regional context.")
    reg_df = pd.DataFrame()

# Working frame — use IDP as-is (no merge needed; all enriched fields present)
df = idp_df.copy()
print(f"\nWorking frame: {len(df):,} rows × {len(df.columns)} cols")


# COMMAND ----------
# MAGIC %md ## 3. Layer 0 — Data Integrity Repair

# COMMAND ----------

def repair_data_integrity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pre-analysis data cleanup:
    1. Clip negative scores to 0
    2. Coerce numerics and booleans
    3. Seed missing stat_anomaly columns with False
    4. Fix is_planning_ready overconfidence
    """
    result = df.copy()

    # ── Clip negative scores ─────────────────────────────────────────────────
    score_cols = [
        "emergency_readiness_score", "critical_care_score",
        "medical_desert_score", "data_completeness_score",
        "capability_confidence", "evidence_weight",
        "ghost_probability_score", "quality_risk_score",
        "clinical_risk_score", "operational_risk_score", "integrity_risk_score",
        "service_richness_score", "infrastructure_completeness_score",
        "referral_complexity_score", "healthcare_maturity_score",
        "clinical_completeness", "location_completeness", "contact_completeness",
        "geo_quality_score", "clinical_complexity_score",
    ]
    for col in score_cols:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(0.0).clip(lower=0.0)

    # ── Numeric columns ──────────────────────────────────────────────────────
    int_cols = [
        "number_doctors_int", "capacity_int", "procedure_count", "equipment_count",
        "capability_count", "specialty_count", "total_stat_anomalies",
        "quality_flag_count", "doc_text_length", "dedup_cluster_size",
        "geo_precision_tier", "specialty_direct_count", "specialty_inferred_count",
        "phone_count",
    ]
    for col in int_cols:
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors="coerce").fillna(0).astype("Int64")

    # ── Boolean coercions ────────────────────────────────────────────────────
    bool_cols = [
        "is_hospital", "is_clinic", "is_ngo", "is_public", "is_private",
        "is_faith_based", "is_government", "is_operational",
        "is_duplicate_survivor", "is_generated_canonical",
        "has_emergency_medicine", "has_surgery", "has_obstetrics", "has_icu",
        "has_radiology", "has_infectious_disease", "has_mental_health", "has_pediatrics",
        "has_procedures", "has_equipment", "has_capabilities", "has_specialties",
        "has_description", "has_contact", "has_physical_address", "has_bare_website_domain",
        "accepts_volunteers_bool", "ngo_serves_ghana",
        "is_rag_ready", "is_search_ready", "is_planning_ready", "is_clinical_ready",
        "capability_is_valid", "geo_contradiction_flag", "geo_region_mismatch",
        "is_teaching_hospital", "is_referral_center", "is_specialist_hospital",
        "is_military_hospital", "organization_category_inferred",
        "has_multiple_phones",
        "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
        "stat_anomaly_clinic_claims_icu", "stat_anomaly_ghost_facility",
        "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth",
    ]
    for col in bool_cols:
        if col in result.columns:
            result[col] = coerce_bool_col(result[col])
        else:
            result[col] = False

    # ── Fix is_planning_ready overconfidence ──────────────────────────────────
    if "is_planning_ready" in result.columns and "data_completeness_score" in result.columns:
        result.loc[result["data_completeness_score"] < 0.40, "is_planning_ready"] = False

    # ── Seed procedure_breadth if missing ────────────────────────────────────
    if "stat_anomaly_procedure_breadth" in result.columns:
        proc  = pd.to_numeric(result["procedure_count"],  errors="coerce").fillna(0)
        equip = pd.to_numeric(result["equipment_count"],  errors="coerce").fillna(0).replace(0, np.nan)
        result["stat_anomaly_procedure_breadth"] = (
            (proc > 3) & (proc / equip > cfg.PROC_EQUIP_RATIO_THRESHOLD)
        ).fillna(False).astype(bool)

    # ── Report ───────────────────────────────────────────────────────────────
    still_neg = {
        c: int((result[c] < 0).sum())
        for c in score_cols if c in result.columns and (result[c] < 0).any()
    }
    print(f"[Layer 0] Integrity repair complete. Shape: {result.shape}")
    print(f"  Remaining negative scores: {still_neg or 'none ✅'}")
    dup_pk = result["pk_unique_id"].duplicated().sum() if "pk_unique_id" in result.columns else 0
    dup_nm = result["name"].duplicated().sum() if "name" in result.columns else 0
    print(f"  Duplicate pk_unique_id: {dup_pk} | duplicate names: {dup_nm}")

    return result


df = repair_data_integrity(df)


# COMMAND ----------
# MAGIC %md ## 4. Merge Regional Context

# COMMAND ----------

def merge_regional_context(df: pd.DataFrame, reg: pd.DataFrame) -> pd.DataFrame:
    """
    Left-join regional summary columns for peer comparison and regional baselines.
    Renames to region_* prefix to avoid collisions.
    """
    if reg.empty or "region_normalised" not in reg.columns:
        print("[Regional] No regional data. Skipping.")
        return df

    reg_map = {
        "avg_completeness":                   "region_avg_completeness",
        "avg_ghost_probability":              "region_avg_ghost_probability",
        "avg_clinical_complexity":            "region_avg_clinical_complexity",
        "avg_evidence_weight":                "region_avg_evidence_weight",
        "avg_quality_risk":                   "region_avg_quality_risk",
        "avg_facility_desert_score":          "region_avg_desert_score",
        "avg_emergency_readiness":            "region_avg_emergency_readiness",
        "avg_critical_care_score":            "region_avg_critical_care_score",
        "avg_service_richness_score":         "region_avg_service_richness",
        "avg_infrastructure_completeness_score": "region_avg_infra_completeness",
        "avg_healthcare_maturity_score":      "region_avg_maturity",
        "medical_desert_score":               "region_medical_desert_score",
        "total_region_anomalies":             "region_total_anomalies",
        "facilities_per_100k":                "region_facilities_per_100k",
        "doctors_per_100k":                   "region_doctors_per_100k",
        "beds_per_100k":                      "region_beds_per_100k",
        "icu_facilities_per_100k":            "region_icu_per_100k",
        "surgery_facilities_per_100k":        "region_surgery_per_100k",
        "maternity_gap_score":                "region_maternity_gap",
        "emergency_gap_score":                "region_emergency_gap",
        "icu_gap_score":                      "region_icu_gap",
        "surgical_access_gap_score":          "region_surgical_gap",
        "critical_specialty_gap_count":       "region_specialty_gap_count",
        "recommended_actions":                "region_recommended_actions",
    }
    cols_present = [c for c in reg_map.keys() if c in reg.columns]
    reg_sub = (
        reg[["region_normalised"] + cols_present]
        .rename(columns={c: reg_map[c] for c in cols_present})
    )
    # Drop any already-present region_ cols to avoid _x / _y suffixes
    existing_region_cols = [c for c in reg_sub.columns if c != "region_normalised" and c in df.columns]
    df = df.drop(columns=existing_region_cols, errors="ignore")
    df = df.merge(reg_sub, on="region_normalised", how="left")
    n_merged = df["region_avg_completeness"].notna().sum() if "region_avg_completeness" in df.columns else 0
    print(f"[Regional] Merged context for {n_merged:,} facilities.")
    return df


df = merge_regional_context(df, reg_df)


# COMMAND ----------
# MAGIC %md ## 5. Layer 1a — Evidence-Absence Confidence

# COMMAND ----------

def compute_evidence_absence_confidence(df: pd.DataFrame) -> pd.DataFrame:
    """
    IDP-native version: uses `evidence_weight` directly from IDP as primary signal.
    Falls back to source_trust + completeness-based estimate when weight is low.

    evidence_weight (IDP field, 0-1) already encodes cross-source reliability.
    We blend it with completeness and source_trust for the final confidence.
    """
    result = df.copy()

    def _absence_confidence(row) -> float:
        completeness   = safe_float(row.get("data_completeness_score"), 0.5)
        source_trust   = safe_str(row.get("source_trust"), "inferred")
        evidence_wt    = safe_float(row.get("evidence_weight"), 0.5)   # IDP-native
        is_bare_domain = safe_bool(row.get("has_bare_website_domain"), False)
        has_social     = any([
            safe_bool(row.get("facebooklink")),
            safe_bool(row.get("twitterlink")),
            safe_bool(row.get("instagramlink")),
        ])

        # Base from IDP evidence_weight (primary signal)
        base = evidence_wt * 0.55

        # Supplement with source trust
        trust_add = {
            "structured":      0.30,
            "text_extracted":  0.20,
            "social_metadata": 0.10,
            "inferred":        0.03,
        }.get(source_trust, 0.08)

        # Completeness modulation
        if completeness >= cfg.HIGH_COMPLETENESS_THRESHOLD:
            comp_factor = 1.15
        elif completeness >= cfg.LOW_COMPLETENESS_THRESHOLD:
            comp_factor = 0.90
        else:
            comp_factor = 0.55

        # Penalties / bonuses
        domain_pen  = 0.80 if is_bare_domain else 1.0
        social_bon  = 1.05 if has_social else 1.0

        conf = (base + trust_add) * comp_factor * domain_pen * social_bon
        return round(min(max(conf, 0.0), 1.0), 4)

    result["evidence_absence_confidence"] = result.apply(_absence_confidence, axis=1)
    print(f"[Layer 1a] Evidence-absence confidence. Mean: {result['evidence_absence_confidence'].mean():.3f}")
    return result


df = compute_evidence_absence_confidence(df)


# COMMAND ----------
# MAGIC %md ## 6. Layer 1b — Duplicate Identity Detection

# COMMAND ----------

def detect_duplicate_identities(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identify near-duplicate facilities (same normalised name + region).
    Marks non-survivors with identity_duplicate_risk = True.
    """
    result = df.copy()
    result["identity_duplicate_flag"]    = False
    result["identity_duplicate_cluster"] = 0
    result["identity_duplicate_risk"]    = False

    if "name" not in result.columns or "region_normalised" not in result.columns:
        return result

    result["_name_norm"] = (
        result["name"].astype(str)
        .str.lower().str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"[^a-z0-9 ]", "", regex=True)
    )
    result["_region_norm"] = result["region_normalised"].astype(str).str.lower().str.strip()
    result["_dedup_key"]   = result["_name_norm"] + "|" + result["_region_norm"]

    dup_keys = result[result["_dedup_key"].duplicated(keep=False)]["_dedup_key"].unique()
    result["identity_duplicate_flag"] = result["_dedup_key"].isin(dup_keys)

    cluster_id_map = {k: i + 1 for i, k in enumerate(dup_keys)}
    result["identity_duplicate_cluster"] = result["_dedup_key"].map(cluster_id_map).fillna(0).astype(int)

    is_survivor = result.get("is_duplicate_survivor", pd.Series(True, index=result.index)).apply(safe_bool)
    result["identity_duplicate_risk"] = result["identity_duplicate_flag"] & (~is_survivor)

    result = result.drop(columns=["_name_norm", "_region_norm", "_dedup_key"], errors="ignore")

    n_dup  = int(result["identity_duplicate_flag"].sum())
    n_risk = int(result["identity_duplicate_risk"].sum())
    print(f"[Layer 1b] Duplicates: {n_dup} flagged | {n_risk} non-survivor risks")
    return result


df = detect_duplicate_identities(df)


# COMMAND ----------
# MAGIC %md ## 7. Layer 1c — Statistical Anomaly Detection (IDP-Native, Evidence-Weighted)

# COMMAND ----------

STAT_FLAG_COLS_EXISTING = [c for c in df.columns if c.startswith("stat_anomaly_")]
print("Pre-existing stat_anomaly flags:")
for col in STAT_FLAG_COLS_EXISTING:
    print(f"  {col:<52} {int(df[col].sum()):>4} flagged")


def run_statistical_anomaly_detection(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    IDP-native, evidence-weighted anomaly detection.

    Key improvements over gold_facilities_enriched version:
    - Uses IDP's `evidence_weight`, `capability_confidence`, `ghost_probability_score`
      directly instead of re-deriving them
    - Leverages `clinical_completeness`, `location_completeness`, `contact_completeness`
      for richer evidence triangulation
    - Capability graph scores (`service_richness_score`, `infrastructure_completeness_score`,
      `healthcare_maturity_score`) inform anomaly plausibility
    - Absence flags respect IDP's own `capability_is_valid` judgement
    - Not overly strict: absence of data ≠ capability failure
    """
    result = df.copy()

    # ── Safe accessors ─────────────────────────────────────────────────────────
    def cb(name: str) -> pd.Series:
        return result.get(name, pd.Series(False, index=result.index)).apply(safe_bool)

    def cn(name: str, d: float = 0.0) -> pd.Series:
        if name in result.columns:
            return pd.to_numeric(result[name], errors="coerce").fillna(d)
        return pd.Series(d, index=result.index)

    is_hospital  = cb("is_hospital")
    is_clinic    = cb("is_clinic")
    is_ngo       = cb("is_ngo")
    is_referral  = cb("is_referral_center")
    is_teaching  = cb("is_teaching_hospital")

    capacity     = cn("capacity_int")
    doctors      = cn("number_doctors_int")
    proc_cnt     = cn("procedure_count")
    equip_cnt    = cn("equipment_count")
    cap_cnt      = cn("capability_count")
    spec_cnt     = cn("specialty_count")

    has_icu      = cb("has_icu")
    has_surg     = cb("has_surgery")
    has_obs      = cb("has_obstetrics")
    has_em       = cb("has_emergency_medicine")

    # IDP-native signals
    completeness  = cn("data_completeness_score", 0.5)
    cap_conf      = cn("capability_confidence", 1.0)       # IDP: validated confidence
    evidence_wt   = cn("evidence_weight", 0.5)             # IDP: cross-source weight
    ghost_prob    = cn("ghost_probability_score", 0.0)     # IDP: ghost probability
    quality_risk  = cn("quality_risk_score", 0.0)          # IDP: overall quality risk
    absence_conf  = cn("evidence_absence_confidence", 0.5) # computed in Layer 1a
    has_contact   = cb("has_contact")
    cap_is_valid  = cb("capability_is_valid")              # IDP: validated capability flag

    # IDP graph scores
    svc_richness   = cn("service_richness_score", 0.0)
    infra_complete = cn("infrastructure_completeness_score", 0.0)
    maturity       = cn("healthcare_maturity_score", 0.0)

    # Completeness components
    clin_complete  = cn("clinical_completeness", 0.5)
    loc_complete   = cn("location_completeness", 0.5)
    cont_complete  = cn("contact_completeness", 0.5)

    # ── [S1] Capability Inflation ──────────────────────────────────────────────
    # IDP improvement: IDP's cap_conf must also support the flag
    cap_equip_ratio = cap_cnt / equip_cnt.replace(0, np.nan)
    result["stat_anomaly_capability_inflation"] = (
        (cap_equip_ratio > cfg.CAPABILITY_INFLATION_RATIO) &
        (cap_cnt >= cfg.CAPABILITY_INFLATION_MIN_CAP) &
        (absence_conf >= 0.48) &
        (completeness >= 0.42) &
        (cap_conf >= 0.60) &           # IDP: only flag when IDP itself is confident
        (~is_ngo)                       # NGOs often have high capability claims legitimately
    ).fillna(False).astype(bool)

    # ── [S2] Hospital With No Doctors (evidence-weighted) ─────────────────────
    # IDP improvement: require clinical_completeness to be high (doctor record should exist)
    result["stat_anomaly_hospital_no_doctors"] = (
        is_hospital &
        (doctors == 0) &
        (completeness >= 0.58) &
        (clin_complete >= 0.50) &      # clinical section is filled, so absence is real
        (absence_conf >= 0.62) &
        (proc_cnt > 0) &               # has operational signals
        (~is_ngo)                      # NGO hospitals often have visiting doctors only
    ).fillna(False).astype(bool)

    # ── [S3] Clinic Claims ICU ────────────────────────────────────────────────
    result["stat_anomaly_clinic_claims_icu"] = (
        is_clinic & has_icu &
        (~is_referral) &              # referral clinics can legitimately have ICU
        (cap_conf >= 0.55)            # IDP must back the capability
    ).fillna(False).astype(bool)

    # ── [S4] Ghost Facility (multi-signal) ────────────────────────────────────
    # IDP improvement: ghost_probability_score is the IDP's own ghost estimate
    result["stat_anomaly_ghost_facility"] = (
        (completeness < cfg.GHOST_COMPLETENESS_CUTOFF) &
        (cap_cnt >= cfg.GHOST_MIN_CAPABILITY_CLAIMS) &
        (equip_cnt == 0) &
        (proc_cnt == 0) &
        (~has_contact) &
        (ghost_prob >= cfg.GHOST_PROB_MIN) &   # IDP ghost score must agree
        (evidence_wt < 0.45)                    # low overall evidence
    ).fillna(False).astype(bool)

    # ── [S5] Specialty Mismatch (IDP-validated) ───────────────────────────────
    # IDP improvement: rely on IDP's own specialty validation via capability_is_valid
    if "stat_anomaly_specialty_mismatch" in result.columns:
        result["stat_anomaly_specialty_mismatch"] = (
            cb("stat_anomaly_specialty_mismatch") &
            (cap_conf >= 0.45) &
            (completeness >= 0.42) &
            (~cap_is_valid)            # IDP flagged capability as invalid
        ).fillna(False).astype(bool)

    # ── [S6] Procedure Breadth (IDP-weighted) ─────────────────────────────────
    proc_equip_ratio = proc_cnt / equip_cnt.replace(0, np.nan)
    result["stat_anomaly_procedure_breadth"] = (
        (proc_cnt > 3) &
        (proc_equip_ratio > cfg.PROC_EQUIP_RATIO_THRESHOLD) &
        (equip_cnt > 0) &
        (absence_conf >= 0.50)
    ).fillna(False).astype(bool)

    # ── [E1] Type-Capability Mismatch ─────────────────────────────────────────
    result["enhanced_type_capability_mismatch"] = (
        is_clinic &
        (has_icu | (has_surg & has_em)) &
        (cap_cnt >= 2) &
        (absence_conf >= 0.43) &
        (~is_referral) &
        (cap_conf >= 0.50)
    ).fillna(False).astype(bool)

    # ── [E2] Ghost Hospital ───────────────────────────────────────────────────
    result["enhanced_ghost_hospital"] = (
        is_hospital &
        (capacity > 80) &
        (doctors == 0) &
        (completeness >= 0.52) &
        (absence_conf >= 0.58) &
        (~is_teaching)                # teaching hospitals may not record all doctors
    ).fillna(False).astype(bool)

    # ── [E3] Procedures Without Equipment ────────────────────────────────────
    result["enhanced_procedures_no_equipment"] = (
        (proc_cnt > 5) &
        (equip_cnt == 0) &
        (completeness >= 0.48) &
        (absence_conf >= 0.52) &
        (~is_ngo)                     # NGO outreach may do procedures without owned equipment
    ).fillna(False).astype(bool)

    # ── [E4] Low IDP Confidence ───────────────────────────────────────────────
    result["enhanced_low_idp_confidence"] = (
        (cap_conf < 0.50) &
        (cap_cnt >= 2) &
        (~cap_is_valid)               # IDP explicitly marked capabilities as invalid
    ).fillna(False).astype(bool)

    # ── [E5] Suspicious Completeness ─────────────────────────────────────────
    result["enhanced_suspicious_completeness"] = (
        (completeness < 0.38) &
        (cap_cnt >= 3) &
        (equip_cnt == 0) &
        (ghost_prob >= 0.35)          # elevated ghost probability supports the flag
    ).fillna(False).astype(bool)

    # ── [E6] ICU Without Infrastructure ──────────────────────────────────────
    result["enhanced_icu_no_infrastructure"] = (
        has_icu &
        (equip_cnt < cfg.ICU_MIN_EQUIPMENT) &
        (cap_cnt < cfg.ICU_MIN_CAPABILITY) &
        (absence_conf >= 0.52) &
        (infra_complete < 0.20)       # IDP's own infrastructure score is very low
    ).fillna(False).astype(bool)

    # ── [E7] Implausible Doctor:Bed Ratio ─────────────────────────────────────
    doc_bed_ratio = doctors / capacity.replace(0, np.nan)
    result["enhanced_implausible_doctor_bed_ratio"] = (
        is_hospital &
        (doctors > 0) &
        (capacity > 0) &
        (doc_bed_ratio > cfg.MAX_DOCTOR_BED_RATIO) &
        (completeness >= 0.50)        # don't flag sparse records
    ).fillna(False).astype(bool)

    # ── [E8] EM Without Surgical Support ─────────────────────────────────────
    result["enhanced_em_without_surgical_support"] = (
        has_em &
        (~has_surg) &
        (~has_obs) &
        is_hospital &
        (equip_cnt == 0) &
        (completeness >= 0.50) &
        (infra_complete < 0.15)       # infrastructure score is near zero
    ).fillna(False).astype(bool)

    # ── [E9] Geo Contradiction (IDP-flagged) ──────────────────────────────────
    result["enhanced_geo_contradiction"] = (
        cb("geo_contradiction_flag") | cb("geo_region_mismatch")
    ).astype(bool)

    # ── [E10] Planning-Ready Overconfidence ───────────────────────────────────
    result["enhanced_planning_overconfidence"] = (
        cb("is_planning_ready") &
        (completeness < 0.43)
    ).fillna(False).astype(bool)

    # ── [E11] Capability Graph Dependency Gap ─────────────────────────────────
    # IDP-native: uses capability_dependency_gaps from capability graph engine
    dep_gaps_count = pd.Series(0, index=result.index)
    if "capability_dependency_gaps" in result.columns:
        dep_gaps_count = result["capability_dependency_gaps"].apply(
            lambda v: len(ensure_list(v))
        )
    result["enhanced_graph_dependency_gap"] = (
        (dep_gaps_count >= 2) &
        (has_icu | has_surg | has_em | has_obs) &
        (infra_complete < 0.50) &
        (completeness >= 0.38) &
        (cap_conf >= 0.45)
    ).fillna(False).astype(bool)

    # ── [E12] Service Richness vs Equipment Mismatch (IDP-native) ─────────────
    # IDP's service_richness_score is HIGH but equipment is absent
    result["enhanced_richness_equipment_mismatch"] = (
        (svc_richness >= cfg.RICHNESS_EQUIP_MIN_RICHNESS) &
        (equip_cnt == 0) &
        (completeness >= 0.50) &
        (absence_conf >= 0.50)
    ).fillna(False).astype(bool)

    # ── [E13] Maturity vs Infrastructure Mismatch (IDP-native) ───────────────
    # High healthcare_maturity_score but very low infrastructure_completeness
    result["enhanced_maturity_infra_mismatch"] = (
        (maturity >= cfg.MATURITY_INFRA_MISMATCH_MAT) &
        (infra_complete < cfg.MATURITY_INFRA_MISMATCH_INFRA) &
        (completeness >= 0.45) &
        (absence_conf >= 0.48)
    ).fillna(False).astype(bool)

    # ── [E14] High Quality Risk (IDP-flagged) ─────────────────────────────────
    # IDP's own quality_risk_score is high and completeness is sufficient
    result["enhanced_high_quality_risk"] = (
        (quality_risk >= 0.55) &
        (completeness >= 0.45) &
        (cap_cnt >= 2)
    ).fillna(False).astype(bool)

    # ── Collect all flag columns ───────────────────────────────────────────────
    ALL_FLAG_DEF = [
        "stat_anomaly_capability_inflation",
        "stat_anomaly_hospital_no_doctors",
        "stat_anomaly_clinic_claims_icu",
        "stat_anomaly_ghost_facility",
        "stat_anomaly_specialty_mismatch",
        "stat_anomaly_procedure_breadth",
        "enhanced_type_capability_mismatch",
        "enhanced_ghost_hospital",
        "enhanced_procedures_no_equipment",
        "enhanced_low_idp_confidence",
        "enhanced_suspicious_completeness",
        "enhanced_icu_no_infrastructure",
        "enhanced_implausible_doctor_bed_ratio",
        "enhanced_em_without_surgical_support",
        "enhanced_geo_contradiction",
        "enhanced_planning_overconfidence",
        "enhanced_graph_dependency_gap",
        "enhanced_richness_equipment_mismatch",
        "enhanced_maturity_infra_mismatch",
        "enhanced_high_quality_risk",
    ]
    # Merge with any pre-existing stat flags from the original IDP table
    all_existing = [c for c in STAT_FLAG_COLS_EXISTING if c not in ALL_FLAG_DEF]
    candidate_cols = all_existing + ALL_FLAG_DEF

    seen = set()
    unique_flag_cols = []
    for c in candidate_cols:
        if c not in seen and c in result.columns:
            seen.add(c)
            unique_flag_cols.append(c)
            result[c] = result[c].astype(bool)

    result["total_anomaly_flags"] = result[unique_flag_cols].astype(int).sum(axis=1)

    print(f"\n[Layer 1c] Statistical anomaly detection complete.")
    print(f"  Active flags across {len(unique_flag_cols)} flag types:")
    for fc in unique_flag_cols:
        n = int(result[fc].sum())
        if n > 0:
            print(f"    {fc:<58}  {n:>4}  ({n/len(result):.1%})")

    return result, unique_flag_cols


df, ALL_FLAG_COLS = run_statistical_anomaly_detection(df)


# COMMAND ----------
# MAGIC %md ## 8. Layer 1d — Peer-Group Comparative Scoring

# COMMAND ----------

def compute_peer_comparison_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compare each facility against same-region + same-type peers.
    High-cap + low-equip outliers = inflated capability signal.
    Uses IDP-native graph scores in the z-score calculation.
    """
    result = df.copy()
    result["peer_capability_zscore"]  = 0.0
    result["peer_procedure_zscore"]   = 0.0
    result["peer_outlier_high_cap"]   = False
    result["peer_outlier_low_equip"]  = False

    if "facility_type_clean" not in result.columns or "region_normalised" not in result.columns:
        print("[Layer 1d] Peer comparison skipped.")
        return result

    def zscore_safe(series: pd.Series) -> pd.Series:
        std = series.std()
        return pd.Series(0.0, index=series.index) if std == 0 else \
               ((series - series.mean()) / std).clip(-5, 5)

    for grp_keys, grp in result.groupby(["region_normalised", "facility_type_clean"], dropna=False):
        if len(grp) < 3:
            continue
        idx   = grp.index
        cap   = grp["capability_count"].astype(float)
        proc  = grp["procedure_count"].astype(float)
        equip = grp["equipment_count"].astype(float)

        result.loc[idx, "peer_capability_zscore"] = zscore_safe(cap).values
        result.loc[idx, "peer_procedure_zscore"]  = zscore_safe(proc).values
        result.loc[idx, "peer_outlier_high_cap"]  = (zscore_safe(cap)  >  cfg.PEER_ZSCORE_THRESHOLD).values
        result.loc[idx, "peer_outlier_low_equip"] = (zscore_safe(equip) < -cfg.PEER_ZSCORE_THRESHOLD).values

    result["enhanced_peer_capability_outlier"] = (
        result["peer_outlier_high_cap"] & result["peer_outlier_low_equip"]
    ).astype(bool)

    if "enhanced_peer_capability_outlier" not in ALL_FLAG_COLS:
        ALL_FLAG_COLS.append("enhanced_peer_capability_outlier")
        result["total_anomaly_flags"] = result[ALL_FLAG_COLS].astype(int).sum(axis=1)

    n_out = int(result["enhanced_peer_capability_outlier"].sum())
    print(f"[Layer 1d] Peer comparison complete. High-cap + low-equip outliers: {n_out}")
    return result


df = compute_peer_comparison_scores(df)


# COMMAND ----------
# MAGIC %md ## 9. Layer 1e — Service Maturity Tiers (IDP-Confidence-Aware)

# COMMAND ----------

def compute_service_maturity(df: pd.DataFrame) -> pd.DataFrame:
    """
    IDP-native maturity tiers using capability_confidence and evidence_weight.

    TIER 5 — CORROBORATED : IDP validated, high evidence weight, multi-source
    TIER 4 — CREDIBLE     : IDP processed, moderate completeness, proc+equip
    TIER 3 — CLAIMED      : Capabilities claimed, basic IDP evidence
    TIER 2 — SPARSE       : Low completeness, minimal signals
    TIER 1 — SHELL        : Very low completeness, no operational signals
    """
    result = df.copy()

    def _maturity(row) -> Tuple[int, str]:
        completeness = safe_float(row.get("data_completeness_score"), 0.0)
        cap_conf     = safe_float(row.get("capability_confidence"),   0.0)
        evidence_wt  = safe_float(row.get("evidence_weight"),         0.0)
        equip_cnt    = safe_int(row.get("equipment_count"))
        proc_cnt     = safe_int(row.get("procedure_count"))
        cap_cnt      = safe_int(row.get("capability_count"))
        has_idp_enrich = bool(row.get("procedure_enriched"))   # has IDP enrichment
        source_trust = safe_str(row.get("source_trust"), "inferred")
        cap_is_valid = safe_bool(row.get("capability_is_valid"), True)
        svc_rich     = safe_float(row.get("service_richness_score"), 0.0)

        # TIER 5: full IDP corroboration
        if (
            completeness >= 0.72 and
            cap_conf >= 0.80 and
            evidence_wt >= 0.70 and
            cap_is_valid and
            (equip_cnt >= 2 or proc_cnt >= 3) and
            (has_idp_enrich or source_trust == "structured")
        ):
            return 5, "CORROBORATED"

        # TIER 4: credible IDP evidence
        if (
            completeness >= 0.52 and
            cap_conf >= 0.65 and
            evidence_wt >= 0.50 and
            (equip_cnt >= 1 or proc_cnt >= 2) and
            cap_cnt >= 2
        ):
            return 4, "CREDIBLE"

        # TIER 3: claimed with basic evidence
        if (
            cap_cnt >= 2 and
            completeness >= 0.38 and
            (cap_conf >= 0.55 or svc_rich >= 0.15)
        ):
            return 3, "CLAIMED"

        # TIER 2: sparse
        if completeness >= 0.28 or cap_cnt >= 1:
            return 2, "SPARSE"

        return 1, "SHELL"

    maturity = result.apply(_maturity, axis=1)
    result["service_maturity_tier"]  = maturity.apply(lambda x: x[0]).astype(int)
    result["service_maturity_label"] = maturity.apply(lambda x: x[1])

    dist = result["service_maturity_label"].value_counts().to_dict()
    print(f"[Layer 1e] Service maturity: {dist}")
    return result


df = compute_service_maturity(df)


# COMMAND ----------
# MAGIC %md ## 10. Layer 1f — Continuity & Fragility Risk

# COMMAND ----------

def compute_continuity_risk(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assess fragility of service delivery:
    - Single-specialist dependency
    - NGO visiting-team dependency
    - Complexity without infrastructure
    - IDP-native: uses clinical_completeness, contact_completeness
    """
    result = df.copy()

    def _continuity(row) -> Tuple[float, str]:
        doctors       = safe_int(row.get("number_doctors_int"))
        cap_cnt       = safe_int(row.get("capability_count"))
        spec_cnt      = safe_int(row.get("specialty_count"))
        is_ngo        = safe_bool(row.get("is_ngo"))
        is_hosp       = safe_bool(row.get("is_hospital"))
        has_surg      = safe_bool(row.get("has_surgery"))
        has_icu       = safe_bool(row.get("has_icu"))
        equip_cnt     = safe_int(row.get("equipment_count"))
        complexity    = safe_str(row.get("facility_complexity_level"), "low")
        maturity_tier = safe_int(row.get("service_maturity_tier"), 2)
        clin_comp     = safe_float(row.get("clinical_completeness"), 0.5)
        infra_score   = safe_float(row.get("infrastructure_completeness_score"), 0.5)

        risk  = 0.0
        flags = []

        # Single-specialist dependency
        if doctors == 1 and (has_surg or has_icu or cap_cnt > 4):
            risk += 0.35
            flags.append("single_specialist_dependency")

        # No doctors + complex claims
        if doctors == 0 and spec_cnt >= 2 and clin_comp >= 0.45:
            risk += 0.28
            flags.append("specialist_absent_complex_claims")

        # NGO visiting-team dependency
        if is_ngo and equip_cnt == 0 and cap_cnt >= 2:
            risk += 0.22
            flags.append("ngo_visiting_team_dependency")

        # High complexity, low maturity
        if complexity in ("high", "complex") and maturity_tier <= 2:
            risk += 0.20
            flags.append("complexity_without_evidence")

        # Hospital with no equipment + significant capability claims
        if is_hosp and equip_cnt == 0 and cap_cnt >= 3 and infra_score < 0.15:
            risk += 0.15
            flags.append("hospital_no_equipment")

        return round(min(risk, 1.0), 4), "|".join(flags)

    cont = result.apply(_continuity, axis=1)
    result["continuity_risk_score"] = cont.apply(lambda x: x[0]).astype(float)
    result["continuity_risk_flags"] = cont.apply(lambda x: x[1])
    result["high_continuity_risk"]  = result["continuity_risk_score"] >= 0.35

    if "high_continuity_risk" not in ALL_FLAG_COLS:
        ALL_FLAG_COLS.append("high_continuity_risk")
        result["total_anomaly_flags"] = result[ALL_FLAG_COLS].astype(int).sum(axis=1)

    n_high = int(result["high_continuity_risk"].sum())
    print(f"[Layer 1f] Continuity risk complete. High-risk: {n_high}")
    return result


df = compute_continuity_risk(df)


# COMMAND ----------
# MAGIC %md ## 11. Layer 1g — Composite Scoring + Data Poverty Classification

# COMMAND ----------

def compute_composite_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    IDP-native composite anomaly score.

    Key design:
    - ghost_probability_score (IDP) is the PRIMARY signal (weight 0.28)
    - quality_risk_score (IDP) is a strong secondary signal
    - flag_norm provides count-based evidence
    - evidence_absence is calibrated by completeness (no low-completeness inflation)
    - Data poverty separation: low completeness + low ghost ≠ anomaly

    Risk taxonomy:
    CRITICAL / HIGH / MEDIUM / LOW / DATA_POVERTY / CLEAN
    """
    result = df.copy()

    flag_count      = result["total_anomaly_flags"].astype(float)
    max_flags       = max(float(flag_count.max()), 1.0)
    flag_norm       = (flag_count / max_flags).clip(0, 1)

    ghost_prob      = pd.to_numeric(result.get("ghost_probability_score",
                                     pd.Series(0.0, index=result.index)), errors="coerce").fillna(0.0)
    quality_risk    = pd.to_numeric(result.get("quality_risk_score",
                                     pd.Series(0.0, index=result.index)), errors="coerce").fillna(0.0)
    absence_conf    = pd.to_numeric(result.get("evidence_absence_confidence",
                                     pd.Series(0.5, index=result.index)), errors="coerce").fillna(0.5)
    continuity_risk = pd.to_numeric(result.get("continuity_risk_score",
                                     pd.Series(0.0, index=result.index)), errors="coerce").fillna(0.0)
    completeness    = pd.to_numeric(result.get("data_completeness_score",
                                     pd.Series(0.5, index=result.index)), errors="coerce").fillna(0.5)

    # Calibrate absence_conf by completeness (avoid penalising poor-data facilities)
    calibrated_absence = (absence_conf * completeness.clip(0.28, 1.0)).round(4)

    result["composite_anomaly_score"] = (
        cfg.W_FLAG_NORM      * flag_norm +
        cfg.W_GHOST_PROB     * ghost_prob +
        cfg.W_QUALITY_RISK   * quality_risk +
        cfg.W_CALIBRATED_ABS * calibrated_absence +
        cfg.W_CONTINUITY     * continuity_risk
    ).clip(0.0, 1.0).round(4)

    # Data poverty: missing data, not a ghost
    result["data_poverty_flag"] = (
        (completeness < cfg.DATA_POVERTY_COMPLETENESS_MAX) &
        (ghost_prob   < cfg.DATA_POVERTY_GHOST_MAX)
    ).astype(bool)

    # ── Risk level assignment ──────────────────────────────────────────────────
    def _risk(row) -> str:
        n          = safe_int(row.get("total_anomaly_flags"))
        comp       = safe_float(row.get("data_completeness_score"), 0.5)
        cscore     = safe_float(row.get("composite_anomaly_score"), 0.0)
        ghost      = safe_float(row.get("ghost_probability_score"), 0.0)
        qual_risk  = safe_float(row.get("quality_risk_score"), 0.0)
        is_poverty = safe_bool(row.get("data_poverty_flag"), False)

        # Data poverty: insufficient data, not evidence of anomaly
        if is_poverty and cscore < 0.58:
            if n >= 2 or cscore >= 0.38:
                return "HIGH"
            if n >= 1 or cscore >= 0.22:
                return "MEDIUM"
            return "DATA_POVERTY"

        # CRITICAL: strong multi-signal evidence
        if (ghost >= 0.72 and cscore >= 0.48):
            return "CRITICAL"
        if cscore >= cfg.RISK_CRITICAL_COMPOSITE and n >= cfg.RISK_CRITICAL_FLAGS_MIN and comp >= cfg.RISK_CRITICAL_COMP_MIN:
            return "CRITICAL"
        if cscore >= 0.72:
            return "CRITICAL"

        if cscore >= cfg.RISK_HIGH_COMPOSITE or n >= 2:
            return "HIGH"
        if cscore >= cfg.RISK_MEDIUM_COMPOSITE or (n == 1 and comp >= 0.52):
            return "MEDIUM"
        if n >= 1:
            return "LOW"
        if ghost >= 0.55 or qual_risk >= 0.50:
            return "MEDIUM"
        return "CLEAN"

    result["anomaly_risk_level"] = result.apply(_risk, axis=1)

    # ── Quality flag taxonomy ──────────────────────────────────────────────────
    def _qual_flags(row) -> str:
        flags = []
        comp         = safe_float(row.get("data_completeness_score"), 0.5)
        source_trust = safe_str(row.get("source_trust"), "inferred")
        absence_c    = safe_float(row.get("evidence_absence_confidence"), 0.5)
        equip_cnt    = safe_int(row.get("equipment_count"))
        cap_cnt      = safe_int(row.get("capability_count"))
        is_ngo       = safe_bool(row.get("is_ngo"))
        geo_flag     = safe_bool(row.get("enhanced_geo_contradiction"))
        dup_flag     = safe_bool(row.get("identity_duplicate_risk"))
        cont_risk    = safe_float(row.get("continuity_risk_score"), 0.0)
        cap_valid    = safe_bool(row.get("capability_is_valid"), True)

        if source_trust == "inferred" or absence_c < 0.28:
            flags.append("LOW_SOURCE_TRUST")
        if comp < 0.32:
            flags.append("LIKELY_MISSING_DATA")
        if source_trust == "text_extracted" and comp < 0.43:
            flags.append("EXTRACTION_ERROR")
        if safe_bool(row.get("enhanced_type_capability_mismatch")) or \
           safe_bool(row.get("stat_anomaly_capability_inflation")):
            flags.append("POSSIBLE_MISREPRESENTATION")
        if not cap_valid and cap_cnt >= 2:
            flags.append("IDP_CAPABILITY_INVALID")
        if geo_flag:
            flags.append("GEO_INCONSISTENCY")
        if dup_flag:
            flags.append("IDENTITY_DUPLICATE")
        if equip_cnt == 0 and cap_cnt >= 2:
            flags.append("WEAK_CLINICAL_EVIDENCE")
        if is_ngo and equip_cnt == 0:
            flags.append("OUTREACH_SERVICE_ONLY")
        if cont_risk >= 0.35:
            flags.append("TEMPORAL_UNCERTAINTY")
        if safe_bool(row.get("data_poverty_flag")):
            flags.append("DATA_POVERTY")

        return "|".join(flags) if flags else "NONE"

    result["quality_flag_taxonomy"] = result.apply(_qual_flags, axis=1)

    # ── Distribution summary ───────────────────────────────────────────────────
    risk_dist = result["anomaly_risk_level"].value_counts().to_dict()
    print(f"\n[Composite] Anomaly Risk Distribution:")
    for level in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "DATA_POVERTY", "CLEAN"]:
        count = risk_dist.get(level, 0)
        bar   = "█" * max(1, count // 10)
        print(f"  {level:<14}  {count:>4}  {bar}")
    print(f"\n  Mean composite_anomaly_score : {result['composite_anomaly_score'].mean():.4f}")
    print(f"  Data poverty flag count      : {result['data_poverty_flag'].sum()}")

    return result


df = compute_composite_scores(df)


# COMMAND ----------
# MAGIC %md ## 12. Regional Prioritization Engine

# COMMAND ----------

def compute_regional_prioritization(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Regional healthcare intervention prioritization.
    Uses IDP-native scores for richer signal than facilities-based version.
    """
    result = df.copy()

    def fcol(name: str, default: float = 0.0) -> pd.Series:
        if name in result.columns:
            return pd.to_numeric(result[name], errors="coerce").fillna(default)
        return pd.Series(default, index=result.index)

    # Facility-level risk signals (normalised 0–1)
    max_desert  = max(fcol("medical_desert_score").max(), 1.0)
    max_equip   = max(fcol("equipment_count").max(), 1.0)

    result["facility_desert_norm"]         = fcol("medical_desert_score").clip(0, 10) / 10.0
    result["facility_emergency_gap_norm"]  = (1.0 - fcol("emergency_readiness_score").clip(0, 1.0)).clip(0, 1)
    result["facility_continuity_norm"]     = fcol("continuity_risk_score").clip(0, 1)
    result["facility_anomaly_norm"]        = fcol("composite_anomaly_score").clip(0, 1)
    result["facility_ghost_norm"]          = fcol("ghost_probability_score").clip(0, 1)   # IDP-native
    result["facility_low_infra_norm"]      = (1.0 - fcol("infrastructure_completeness_score").clip(0, 1)).clip(0, 1)
    result["facility_low_equip_norm"]      = (1.0 - fcol("equipment_count") / max_equip).clip(0, 1)
    result["facility_low_maturity_norm"]   = (1.0 - fcol("healthcare_maturity_score").clip(0, 1)).clip(0, 1)

    if "region_doctors_per_100k" in result.columns:
        max_doc = max(fcol("region_doctors_per_100k").max(), 1.0)
        result["facility_low_staff_norm"] = (1.0 - fcol("region_doctors_per_100k") / max_doc).clip(0, 1)
    else:
        result["facility_low_staff_norm"] = 0.5

    # Regional aggregation
    regional = (
        result.groupby("region_normalised", dropna=False)
        .agg(
            facility_count               = ("unique_id", "count"),
            avg_desert_score             = ("facility_desert_norm",        "mean"),
            avg_emergency_gap            = ("facility_emergency_gap_norm", "mean"),
            avg_continuity_fragility     = ("facility_continuity_norm",    "mean"),
            avg_anomaly_density          = ("facility_anomaly_norm",       "mean"),
            avg_ghost_density            = ("facility_ghost_norm",         "mean"),   # IDP-native
            avg_low_infra_density        = ("facility_low_infra_norm",     "mean"),   # IDP-native
            avg_low_staff_density        = ("facility_low_staff_norm",     "mean"),
            avg_low_equipment_density    = ("facility_low_equip_norm",     "mean"),
            avg_low_maturity_density     = ("facility_low_maturity_norm",  "mean"),   # IDP-native
            critical_facility_count      = ("anomaly_risk_level",          lambda x: (x == "CRITICAL").sum()),
            high_risk_facility_count     = ("anomaly_risk_level",          lambda x: (x == "HIGH").sum()),
            high_continuity_risk_count   = ("high_continuity_risk",        "sum"),
            avg_emergency_readiness      = ("emergency_readiness_score",   "mean"),
            avg_data_completeness        = ("data_completeness_score",     "mean"),
        )
        .reset_index()
    )

    # Priority score (IDP-native ghost and infra signals added)
    regional["regional_priority_score"] = (
        0.25 * regional["avg_desert_score"] +
        0.18 * regional["avg_emergency_gap"] +
        0.15 * regional["avg_ghost_density"] +      # IDP-native ghost signal
        0.14 * regional["avg_low_infra_density"] +  # IDP-native infrastructure gap
        0.12 * regional["avg_continuity_fragility"] +
        0.08 * regional["avg_anomaly_density"] +
        0.05 * regional["avg_low_staff_density"] +
        0.03 * regional["avg_low_maturity_density"]
    ).clip(0, 1).round(4)

    def _tier(score: float) -> str:
        if score >= 0.72: return "P0"
        if score >= 0.58: return "P1"
        if score >= 0.40: return "P2"
        return "P3"

    regional["priority_tier"] = regional["regional_priority_score"].apply(_tier)

    def _interventions(row) -> str:
        recs      = []
        desert    = safe_float(row.get("avg_desert_score"))
        em_gap    = safe_float(row.get("avg_emergency_gap"))
        ghost     = safe_float(row.get("avg_ghost_density"))
        infra_gap = safe_float(row.get("avg_low_infra_density"))
        cont      = safe_float(row.get("avg_continuity_fragility"))
        anomaly   = safe_float(row.get("avg_anomaly_density"))
        staff_gap = safe_float(row.get("avg_low_staff_density"))
        equip_gap = safe_float(row.get("avg_low_equipment_density"))
        completeness = safe_float(row.get("avg_data_completeness"))
        crit_n    = safe_int(row.get("critical_facility_count"))
        high_n    = safe_int(row.get("high_risk_facility_count"))

        if desert >= 0.70:
            recs.append("URGENT: Expand regional access via mobile clinics and referral expansion")
        elif desert >= 0.50:
            recs.append("Increase primary-care coverage to reduce medical-desert burden")

        if em_gap >= 0.72:
            recs.append("CRITICAL: Strengthen emergency stabilization, ambulance routing, triage")
        elif em_gap >= 0.55:
            recs.append("Improve emergency-care infrastructure and referral escalation pathways")

        if ghost >= 0.60:
            recs.append("Conduct facility verification audit — high ghost-facility probability cluster")
        elif ghost >= 0.40:
            recs.append("Perform targeted operational verification of flagged facilities")

        if infra_gap >= 0.72:
            recs.append("CRITICAL infrastructure gap: deploy diagnostic, surgical, emergency equipment")
        elif infra_gap >= 0.50:
            recs.append("Improve infrastructure readiness through targeted equipment procurement")

        if staff_gap >= 0.70:
            recs.append("URGENT workforce shortage: deploy rotating clinicians and specialist outreach")
        elif staff_gap >= 0.50:
            recs.append("Increase staffing density via regional recruitment and NGO workforce programs")

        if cont >= 0.65:
            recs.append("Reduce continuity fragility: build permanent specialist coverage")
        elif cont >= 0.48:
            recs.append("Improve continuity-of-care resilience and specialist coverage stability")

        if anomaly >= 0.65:
            recs.append("Urgent capability-claims audit for inconsistent facility data")

        if completeness <= 0.35:
            recs.append("Prioritize field data collection — low evidence coverage across region")

        if crit_n >= 4:
            recs.append("Multiple critical-risk facilities: coordinated intervention planning required")
        elif high_n >= 3:
            recs.append("Elevated high-risk facility concentration detected")

        if em_gap >= 0.48 and cont >= 0.42:
            recs.append("Strengthen inter-facility referral and emergency transfer pathways")

        if staff_gap >= 0.58 and equip_gap >= 0.58:
            recs.append("Consider NGO mobile outreach and rotating specialist deployment programs")

        if not recs:
            recs.append("Maintain operational monitoring and periodic healthcare-capacity assessment")

        seen, out = set(), []
        for r in recs:
            if r not in seen:
                out.append(r)
                seen.add(r)
        return " | ".join(out[:8])

    regional["recommended_interventions"] = regional.apply(_interventions, axis=1)

    # Merge back priority fields to facility table
    merge_cols = ["region_normalised", "regional_priority_score", "priority_tier", "recommended_interventions"]
    result = result.merge(regional[merge_cols], on="region_normalised", how="left")

    # Logging
    print("\n[Regional Prioritization] COMPLETE")
    tier_dist = regional["priority_tier"].value_counts().to_dict()
    for tier in ["P0", "P1", "P2", "P3"]:
        print(f"  {tier}: {tier_dist.get(tier, 0)}")
    print("\nTop Priority Regions:")
    for _, row in regional.sort_values("regional_priority_score", ascending=False).head(8).iterrows():
        print(f"  {row['region_normalised']:<22} score={row['regional_priority_score']:.3f} tier={row['priority_tier']}")

    return result, regional


df, regional_priority_df = compute_regional_prioritization(df)


# COMMAND ----------
# MAGIC %md ## 13. Layer 2 — LLM Validation + Web Search Enrichment

# COMMAND ----------

LLM_ANOMALY_SYSTEM = """You are a senior healthcare intelligence expert specialising in:
- Hospital capability validation for sub-Saharan Africa (Ghana)
- Low-resource healthcare system assessment
- NGO outreach and mobile medicine operations
- IDP (Intelligent Document Parsing) validation
- Medical facility anomaly detection
- Clinical plausibility assessment

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GHANA HEALTHCARE CONTEXT (CRITICAL)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Missing data ≠ missing capability (many facilities have poor digital records)
- NGO facilities often use rotating/visiting specialists — no permanent staff required
- Equipment is frequently under-documented in small and rural facilities
- Outreach surgical camps legitimately provide advanced procedures temporarily
- Clinics claiming ICU is genuinely rare and suspicious
- Ghost facilities (high ghost_probability_score) warrant serious verification
- Low infrastructure_completeness_score from IDP's graph engine = real concern
- High service_richness_score + zero equipment = likely data over-claim
- High healthcare_maturity_score + very low infrastructure = IDP-detected inconsistency

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IDP SIGNAL INTERPRETATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- ghost_probability_score > 0.70: HIGH concern — facility may not exist as described
- capability_confidence < 0.50: IDP could not validate the capability claims
- infrastructure_completeness_score < 0.15: severe infrastructure gap
- service_richness_score > 0.60 + equipment = 0: very suspicious
- capability_is_valid = False: IDP explicitly rejected capability claims
- capability_dependency_gaps: lists specific missing infrastructure for claimed services

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT (JSON ONLY, NO MARKDOWN)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{
  "confirmed_anomaly_count": <int>,
  "anomaly_severity": "CRITICAL" | "HIGH" | "MEDIUM" | "LOW" | "FALSE_POSITIVE",
  "data_quality_score": <float 0.0-10.0>,
  "clinical_plausibility_score": <float 0.0-10.0>,
  "operational_credibility_score": <float 0.0-10.0>,
  "evidence_strength_score": <float 0.0-10.0>,
  "priority_action": "<max 200 chars — specific, actionable>",
  "clinical_assessment": "<1-3 sentences — operational healthcare interpretation>",
  "false_positive_reason": "<concise or null>",
  "primary_risk_driver": "<main factor driving this anomaly>",
  "recommended_quality_category": "<IDP/SOURCE/OPERATIONAL/INFRASTRUCTURE/STAFFING/GHOST>",
  "requires_manual_review": <true|false>,
  "confidence_level": "HIGH" | "MEDIUM" | "LOW"
}

SCORING: 10 = highly reliable; 0 = completely unreliable.
Be conservative on CRITICAL — require multiple corroborating signals."""


def build_anomaly_context(row: pd.Series, web_info: str = "", llm_knowledge: Dict = {}) -> str:
    """Build enriched facility context for LLM validation."""
    specs    = ensure_list(row.get("specialties_enriched")) or ensure_list(row.get("specialties_parsed"))
    procs    = ensure_list(row.get("procedure_enriched"))   or ensure_list(row.get("procedure_parsed"))
    equips   = ensure_list(row.get("equipment_enriched"))   or ensure_list(row.get("equipment_parsed"))
    caps     = ensure_list(row.get("capability_enriched"))  or ensure_list(row.get("capability_parsed"))
    cap_anom = ensure_list(row.get("capability_anomalies"))
    dep_gaps = ensure_list(row.get("capability_dependency_gaps"))
    idp_cits = ensure_list(row.get("idp_citations"))

    # Parse capability graph summary
    graph_summ = parse_capability_graph_summary(row.get("capability_graph_summary", ""))
    active_domains = ensure_list(graph_summ.get("active_domains", []))
    missing_deps   = ensure_list(graph_summ.get("missing_critical_dependencies", []))
    met_deps       = safe_int(graph_summ.get("met_dependencies", 0))
    req_deps       = safe_int(graph_summ.get("required_dependencies", 0))

    # Active flags
    flag_descriptions = {
        "stat_anomaly_capability_inflation":
            f"CAPABILITY INFLATION: {safe_int(row.get('capability_count'))} caps / "
            f"{safe_int(row.get('equipment_count'))} equipment (>{cfg.CAPABILITY_INFLATION_RATIO:.1f}x ratio)",
        "stat_anomaly_hospital_no_doctors":
            "HOSPITAL WITHOUT DOCTORS: Hospital with 0 doctors (clinical_completeness sufficient)",
        "stat_anomaly_clinic_claims_icu":
            "CLINIC CLAIMS ICU: Clinic-tier claiming intensive care",
        "stat_anomaly_ghost_facility":
            f"GHOST FACILITY: completeness={safe_float(row.get('data_completeness_score')):.2f}, "
            f"ghost_prob={safe_float(row.get('ghost_probability_score')):.2f}, no contact/equipment",
        "stat_anomaly_specialty_mismatch":
            "SPECIALTY MISMATCH: Specialty unsupported by IDP capability validation",
        "stat_anomaly_procedure_breadth":
            f"PROCEDURE BREADTH: {safe_int(row.get('procedure_count'))} procs / "
            f"{safe_int(row.get('equipment_count'))} equip",
        "enhanced_type_capability_mismatch":
            "TYPE-CAPABILITY MISMATCH: Clinic claiming hospital-tier ICU/Surgery",
        "enhanced_ghost_hospital":
            f"GHOST HOSPITAL: {safe_int(row.get('capacity_int'))} beds, 0 doctors",
        "enhanced_procedures_no_equipment":
            f"PROCEDURES WITHOUT EQUIPMENT: {safe_int(row.get('procedure_count'))} procs, 0 equip",
        "enhanced_low_idp_confidence":
            f"LOW IDP CONFIDENCE: cap_conf={safe_float(row.get('capability_confidence')):.2f}, "
            f"capability_is_valid=False",
        "enhanced_suspicious_completeness":
            "SUSPICIOUS COMPLETENESS: Low completeness + capability claims + elevated ghost_prob",
        "enhanced_icu_no_infrastructure":
            f"ICU WITHOUT INFRASTRUCTURE: infra_score={safe_float(row.get('infrastructure_completeness_score')):.2f}",
        "enhanced_implausible_doctor_bed_ratio":
            f"IMPLAUSIBLE DOCTOR:BED: {safe_int(row.get('number_doctors_int'))}/"
            f"{safe_int(row.get('capacity_int'))} beds",
        "enhanced_em_without_surgical_support":
            "EM WITHOUT SURGICAL BACKUP: EM + no surgery/obstetrics + zero infrastructure",
        "enhanced_geo_contradiction":
            "GEO CONTRADICTION: IDP-detected coordinate/region mismatch",
        "enhanced_planning_overconfidence":
            "PLANNING OVERCONFIDENCE: is_planning_ready=True but completeness < 0.43",
        "enhanced_graph_dependency_gap":
            f"GRAPH DEPENDENCY GAP: {len(dep_gaps)} missing deps for claimed critical services",
        "enhanced_richness_equipment_mismatch":
            f"RICHNESS/EQUIPMENT MISMATCH: service_richness={safe_float(row.get('service_richness_score')):.2f}, equip=0",
        "enhanced_maturity_infra_mismatch":
            f"MATURITY/INFRA MISMATCH: maturity={safe_float(row.get('healthcare_maturity_score')):.2f}, "
            f"infra={safe_float(row.get('infrastructure_completeness_score')):.2f}",
        "enhanced_high_quality_risk":
            f"HIGH QUALITY RISK (IDP): quality_risk_score={safe_float(row.get('quality_risk_score')):.2f}",
        "enhanced_peer_capability_outlier":
            f"PEER OUTLIER: cap z-score={safe_float(row.get('peer_capability_zscore')):.1f}",
        "high_continuity_risk":
            f"CONTINUITY RISK: {safe_str(row.get('continuity_risk_flags',''))} "
            f"(score={safe_float(row.get('continuity_risk_score')):.2f})",
        "identity_duplicate_risk":
            "IDENTITY DUPLICATE: Non-survivor duplicate in same region",
    }

    active_flags = [
        f"  • {desc}" for flag, desc in flag_descriptions.items()
        if safe_bool(row.get(flag, False))
    ]

    # Regional context
    region_ctx = ""
    if row.get("region_avg_quality_risk"):
        region_ctx = (
            f"\nREGIONAL CONTEXT ({safe_str(row.get('region_normalised','?'))}):\n"
            f"  avg_completeness: {safe_float(row.get('region_avg_completeness')):.2f} | "
            f"avg_ghost_prob: {safe_float(row.get('region_avg_ghost_probability')):.2f} | "
            f"avg_desert: {safe_float(row.get('region_avg_desert_score')):.2f} | "
            f"avg_maturity: {safe_float(row.get('region_avg_maturity')):.2f}"
        )

    # Web info
    web_ctx = f"\nWEB RESEARCH:\n{web_info}" if web_info else ""

    # LLM knowledge
    know_ctx = ""
    if llm_knowledge and isinstance(llm_knowledge, dict):
        know_ctx = (
            f"\nLLM KNOWLEDGE CHECK:\n"
            f"  Known facility : {llm_knowledge.get('is_known_facility','?')}\n"
            f"  Profile        : {llm_knowledge.get('facility_profile','?')[:150]}\n"
            f"  Flag plausibility: {llm_knowledge.get('flag_plausibility','?')}\n"
            f"  Special context  : {llm_knowledge.get('special_context','')[:100]}"
        )

    # IDP trace excerpt
    idp_trace = safe_str(row.get("idp_trace", ""))[:200]
    idp_run   = safe_str(row.get("idp_run_id", ""))

    return (
        f"FACILITY: {safe_str(row.get('name','Unknown'))}\n"
        f"TYPE: {safe_str(row.get('facility_type_clean','?'))} "
        f"[Tier: {safe_str(row.get('facility_tier_label', row.get('service_maturity_label','?')))}]\n"
        f"OPERATOR: {safe_str(row.get('operatorTypeId','?'))} | "
        f"NGO: {safe_bool(row.get('is_ngo'))} | "
        f"TEACHING: {safe_bool(row.get('is_teaching_hospital'))}\n"
        f"LOCATION: {safe_str(row.get('city_clean','?'))}, "
        f"{safe_str(row.get('region_normalised','?'))}, Ghana\n"
        f"SOURCE TRUST: {safe_str(row.get('source_trust','?'))} | "
        f"MATURITY: {safe_str(row.get('service_maturity_label','?'))} "
        f"(tier {safe_int(row.get('service_maturity_tier',0))})\n\n"
        f"━━ IDP SCORES ━━\n"
        f"  completeness       : {safe_float(row.get('data_completeness_score')):.3f}\n"
        f"  evidence_weight    : {safe_float(row.get('evidence_weight')):.3f}\n"
        f"  absence_confidence : {safe_float(row.get('evidence_absence_confidence')):.3f}\n"
        f"  ghost_probability  : {safe_float(row.get('ghost_probability_score')):.3f}\n"
        f"  capability_confidence: {safe_float(row.get('capability_confidence')):.3f}\n"
        f"  capability_is_valid: {safe_bool(row.get('capability_is_valid'))}\n"
        f"  quality_risk_score : {safe_float(row.get('quality_risk_score')):.3f}\n"
        f"  clinical_risk      : {safe_float(row.get('clinical_risk_score')):.3f}\n"
        f"  clinical_completeness: {safe_float(row.get('clinical_completeness')):.3f}\n\n"
        f"━━ IDP GRAPH SCORES ━━\n"
        f"  service_richness         : {safe_float(row.get('service_richness_score')):.3f}\n"
        f"  infrastructure_completeness: {safe_float(row.get('infrastructure_completeness_score')):.3f}\n"
        f"  healthcare_maturity      : {safe_float(row.get('healthcare_maturity_score')):.3f}\n"
        f"  referral_complexity      : {safe_float(row.get('referral_complexity_score')):.3f}\n"
        f"  graph active_domains     : {', '.join(active_domains[:6]) or 'none'}\n"
        f"  graph met_deps/req_deps  : {met_deps}/{req_deps}\n"
        f"  missing_critical_deps    : {', '.join(missing_deps[:5]) or 'none'}\n"
        f"  capability_dependency_gaps: {', '.join(dep_gaps[:6]) or 'none'}\n\n"
        f"━━ CLINICAL DATA ━━\n"
        f"  DOCTORS: {safe_int(row.get('number_doctors_int'))} | "
        f"BEDS: {safe_int(row.get('capacity_int'))} | "
        f"PROCEDURES: {safe_int(row.get('procedure_count'))} | "
        f"EQUIPMENT: {safe_int(row.get('equipment_count'))} | "
        f"CAPABILITIES: {safe_int(row.get('capability_count'))}\n"
        f"  SPECIALTIES ({safe_int(row.get('specialty_count'))}): {', '.join(specs[:6]) or 'None'}\n"
        f"  PROCEDURES: {'; '.join(procs[:5]) or 'None'}\n"
        f"  EQUIPMENT: {'; '.join(equips[:5]) or 'None'}\n"
        f"  CAPABILITIES: {'; '.join(caps[:5]) or 'None'}\n"
        f"  IDP ANOMALIES: {'; '.join(cap_anom) or 'None'}\n"
        f"  IDP RUN: {idp_run}\n\n"
        f"━━ COMPOSITE SCORE ━━\n"
        f"  composite_anomaly_score: {safe_float(row.get('composite_anomaly_score')):.3f}\n"
        f"  continuity_risk: {safe_float(row.get('continuity_risk_score')):.3f} "
        f"[{safe_str(row.get('continuity_risk_flags','none'))}]\n"
        f"  data_poverty_flag: {safe_bool(row.get('data_poverty_flag'))}\n"
        f"{region_ctx}"
        f"{web_ctx}"
        f"{know_ctx}\n\n"
        f"━━ ACTIVE ANOMALY FLAGS ({len(active_flags)}) ━━\n"
        + ("\n".join(active_flags) if active_flags else "  None triggered")
    )


def run_llm_anomaly_validation(
    df:             pd.DataFrame,
    max_rows:       int   = cfg.MAX_LLM_BATCH,
    min_score:      float = cfg.LLM_MIN_SCORE,
    parent_run_id:  Optional[str] = None,
    use_web_search: bool  = cfg.WEB_SEARCH_ENABLED,
    use_llm_knowledge: bool = True,
) -> pd.DataFrame:
    """
    Priority-batched LLM validation with optional web search and LLM knowledge augmentation.

    Selection: top-N by composite_anomaly_score above min_score.
    Enhancement: web-search + LLM knowledge lookup per facility (gracefully degraded).
    """
    eligible   = df[df["composite_anomaly_score"] >= min_score].copy()
    to_process = eligible.nlargest(max_rows, "composite_anomaly_score").index

    print(f"\n[Layer 2 LLM] Processing {len(to_process)} facilities "
          f"(score≥{min_score}, max={max_rows})")
    print(f"  Web search: {'enabled' if use_web_search else 'disabled'} | "
          f"LLM knowledge: {'enabled' if use_llm_knowledge else 'disabled'}")

    llm_cols = [
        "llm_priority_action", "llm_data_quality_score", "llm_confirmed_anomaly_count",
        "llm_anomaly_severity", "llm_clinical_assessment", "llm_false_positive_reason",
        "llm_recommended_quality_category",
    ]
    for col in llm_cols:
        if col not in df.columns:
            df[col] = None

    for i, idx in enumerate(to_process):
        row  = df.loc[idx]
        name = safe_str(row.get("name", "?"))
        rgn  = safe_str(row.get("region_normalised", "?"))
        ftype= safe_str(row.get("facility_type_clean", "?"))

        if i % 10 == 0:
            print(f"  [{i+1:>3}/{len(to_process)}] {name[:58]}")

        # ── Web search enrichment ──────────────────────────────────────────────
        web_info = ""
        if use_web_search:
            try:
                web_info = web_search_facility(name, rgn, ftype)
            except Exception:
                web_info = ""

        # ── LLM knowledge lookup ───────────────────────────────────────────────
        llm_know = {}
        if use_llm_knowledge and int(row.get("total_anomaly_flags", 0)) >= 2:
            active_flag_names = [
                c for c in ALL_FLAG_COLS
                if c in row.index and safe_bool(row.get(c, False))
            ]
            try:
                llm_know = llm_knowledge_search(name, rgn, " | ".join(active_flag_names[:5]))
            except Exception:
                llm_know = {}

        # ── Build context + LLM call ───────────────────────────────────────────
        context = build_anomaly_context(row, web_info=web_info, llm_knowledge=llm_know)
        raw     = call_llama(
            [{"role": "user", "content": context}],
            system_prompt=LLM_ANOMALY_SYSTEM,
            max_tokens=1200,
            temperature=0.05,
        )
        parsed  = parse_json_llm(raw)

        df.at[idx, "llm_priority_action"]              = safe_str(str(parsed.get("priority_action", "")))[:500]
        df.at[idx, "llm_data_quality_score"]           = safe_float(parsed.get("data_quality_score"), 5.0)
        df.at[idx, "llm_confirmed_anomaly_count"]      = safe_int(parsed.get("confirmed_anomaly_count"), 0)
        df.at[idx, "llm_anomaly_severity"]             = safe_str(str(parsed.get("anomaly_severity", "UNKNOWN")))
        df.at[idx, "llm_clinical_assessment"]          = safe_str(str(parsed.get("clinical_assessment", "")))[:500]
        df.at[idx, "llm_false_positive_reason"]        = safe_str(str(parsed.get("false_positive_reason", "")))[:400]
        df.at[idx, "llm_recommended_quality_category"] = safe_str(str(parsed.get("recommended_quality_category", "")))

        # ── MLflow child run ───────────────────────────────────────────────────
        if parent_run_id:
            try:
                with mlflow.start_run(nested=True, run_name=f"llm_{name[:28]}"):
                    mlflow.set_tag("facility_name",    name)
                    mlflow.set_tag("region",           rgn)
                    mlflow.set_tag("risk_level",       safe_str(row.get("anomaly_risk_level")))
                    mlflow.set_tag("web_search_used",  str(bool(web_info)))
                    mlflow.set_tag("llm_know_used",    str(bool(llm_know)))
                    mlflow.log_text(context, "anomaly_context.txt")
                    mlflow.log_text(raw,     "llm_response.txt")
                    mlflow.log_dict(parsed,  "llm_parsed.json")
                    if web_info:
                        mlflow.log_text(web_info, "web_research.txt")
                    if llm_know:
                        mlflow.log_dict(llm_know, "llm_knowledge.json")
                    mlflow.log_metric("data_quality_score",     safe_float(parsed.get("data_quality_score"), 5.0))
                    mlflow.log_metric("confirmed_anomaly_count",safe_int(parsed.get("confirmed_anomaly_count"), 0))
                    mlflow.log_metric("composite_anomaly_score",safe_float(row.get("composite_anomaly_score"), 0.0))
                    mlflow.log_metric("ghost_probability",       safe_float(row.get("ghost_probability_score"), 0.0))
            except Exception:
                pass

        time.sleep(0.6)

    return df


# COMMAND ----------
# MAGIC %md ## 14. Run LLM Validation with MLflow Tracking

# COMMAND ----------

mlflow.set_experiment(cfg.MLFLOW_EXP)

with mlflow.start_run(run_name="08_anomaly_detection_v4_idp_native") as main_run:
    mlflow.set_tag("source_table",  cfg.IDP_TABLE)
    mlflow.set_tag("version",       "v4-idp-native")
    mlflow.set_tag("primary_table", "gold_idp_enriched")
    mlflow.set_tag("web_search",    str(cfg.WEB_SEARCH_ENABLED))
    mlflow.log_metric("total_facilities", len(df))
    mlflow.log_metric("stat_flagged",     int((df["total_anomaly_flags"] >= 1).sum()))
    mlflow.log_metric("critical_risk",    int((df["anomaly_risk_level"] == "CRITICAL").sum()))
    mlflow.log_metric("high_risk",        int((df["anomaly_risk_level"] == "HIGH").sum()))
    mlflow.log_metric("data_poverty",     int((df["anomaly_risk_level"] == "DATA_POVERTY").sum()))

    df = run_llm_anomaly_validation(
        df,
        max_rows          = cfg.MAX_LLM_BATCH,
        min_score         = cfg.LLM_MIN_SCORE,
        parent_run_id     = main_run.info.run_id,
        use_web_search    = cfg.WEB_SEARCH_ENABLED,
        use_llm_knowledge = True,
    )

    llm_processed = int(df["llm_priority_action"].notna().sum())
    llm_confirmed = int(df["llm_confirmed_anomaly_count"].fillna(0).astype(int).sum())
    mlflow.log_metric("llm_processed",      llm_processed)
    mlflow.log_metric("llm_confirmed_total",llm_confirmed)
    avg_dq = df["llm_data_quality_score"].dropna().mean()
    if not (avg_dq != avg_dq):  # not NaN
        mlflow.log_metric("avg_data_quality_score", float(avg_dq))

    print(f"\n✅ MLflow main run: {main_run.info.run_id}")


# COMMAND ----------
# MAGIC %md ## 15. Assemble Anomaly Output Table

# COMMAND ----------

# ── Define output column groups ────────────────────────────────────────────────
BASE_COLS = [
    "unique_id", "name", "city_clean", "region_normalised",
    "facility_type_clean", "facility_tier_label", "service_maturity_label", "service_maturity_tier",
    "organization_type_clean", "operatorTypeId", "source_trust",
    "latitude", "longitude", "geo_quality_score",
    "number_doctors_int", "capacity_int",
    "data_completeness_score", "evidence_absence_confidence",
    "capability_confidence", "capability_is_valid",
    "ghost_probability_score", "ghost_review_priority",
    "quality_risk_score", "clinical_risk_score", "operational_risk_score", "integrity_risk_score",
    "emergency_readiness_score", "critical_care_score",
    "rag_quality_score", "is_planning_ready", "is_search_ready", "is_clinical_ready",
    # IDP-native completeness components
    "clinical_completeness", "location_completeness", "contact_completeness",
    # IDP-native evidence
    "evidence_weight",
]
SPECIALTY_BOOL_COLS = [
    "has_emergency_medicine", "has_surgery", "has_icu", "has_obstetrics",
    "has_radiology", "has_infectious_disease", "has_mental_health", "has_pediatrics",
    "is_hospital", "is_clinic", "is_ngo", "is_faith_based",
    "is_teaching_hospital", "is_referral_center",
]
COUNT_COLS = [
    "procedure_count", "equipment_count", "capability_count", "specialty_count",
]
ANOMALY_SCORE_COLS = [
    "total_anomaly_flags", "composite_anomaly_score", "anomaly_risk_level",
    "data_poverty_flag",
    "continuity_risk_score", "continuity_risk_flags", "high_continuity_risk",
    "identity_duplicate_flag", "identity_duplicate_risk", "identity_duplicate_cluster",
    "peer_capability_zscore", "peer_procedure_zscore",
    "peer_outlier_high_cap", "peer_outlier_low_equip",
    "quality_flag_taxonomy",
]
GRAPH_COLS = [
    "capability_dependency_gaps", "capability_graph_summary",
    "service_richness_score", "infrastructure_completeness_score",
    "referral_complexity_score", "healthcare_maturity_score",
    "capability_graph_version",
]
STAT_FLAG_COLS  = [c for c in df.columns if c.startswith("stat_anomaly_")]
ENHANCED_COLS   = [c for c in df.columns if c.startswith("enhanced_")]
LLM_COLS = [
    "llm_priority_action", "llm_data_quality_score", "llm_confirmed_anomaly_count",
    "llm_anomaly_severity", "llm_clinical_assessment", "llm_false_positive_reason",
    "llm_recommended_quality_category",
]
ENRICHED_COLS = [
    "specialties_enriched", "procedure_enriched", "equipment_enriched",
    "capability_enriched", "capability_anomalies",
]
DESERT_COLS = ["medical_desert_score", "desert_label"]

all_output_cols = (
    [c for c in BASE_COLS        if c in df.columns] +
    [c for c in SPECIALTY_BOOL_COLS if c in df.columns] +
    [c for c in COUNT_COLS       if c in df.columns] +
    [c for c in ANOMALY_SCORE_COLS if c in df.columns] +
    [c for c in GRAPH_COLS       if c in df.columns] +
    [c for c in STAT_FLAG_COLS   if c in df.columns] +
    [c for c in ENHANCED_COLS    if c in df.columns] +
    [c for c in LLM_COLS         if c in df.columns] +
    [c for c in ENRICHED_COLS    if c in df.columns] +
    [c for c in DESERT_COLS      if c in df.columns]
)

# Deduplicate preserving order
seen = set()
final_cols = []
for c in all_output_cols:
    if c not in seen:
        seen.add(c)
        final_cols.append(c)

anomaly_output = (
    df[final_cols]
    .sort_values("composite_anomaly_score", ascending=False)
    .reset_index(drop=True)
)

# ── Type cleanup ───────────────────────────────────────────────────────────────
# Stringify JSON/array columns for Delta compatibility
for col in ENRICHED_COLS + ["capability_anomalies", "continuity_risk_flags", "capability_dependency_gaps"]:
    if col in anomaly_output.columns:
        anomaly_output[col] = anomaly_output[col].apply(
            lambda v: json.dumps(ensure_list(v)) if not isinstance(v, str) else (v or "[]")
        )

# Boolean columns
all_bool_cols = SPECIALTY_BOOL_COLS + STAT_FLAG_COLS + ENHANCED_COLS + [
    "capability_is_valid", "high_continuity_risk", "data_poverty_flag",
    "identity_duplicate_flag", "identity_duplicate_risk",
    "peer_outlier_high_cap", "peer_outlier_low_equip",
]
for c in all_bool_cols:
    if c in anomaly_output.columns:
        anomaly_output[c] = coerce_bool_col(anomaly_output[c])

# Integer columns
int_out_cols = [
    "service_maturity_tier", "total_anomaly_flags", "identity_duplicate_cluster",
    "llm_confirmed_anomaly_count", "procedure_count", "equipment_count",
    "capability_count", "specialty_count",
]
for c in int_out_cols:
    if c in anomaly_output.columns:
        anomaly_output[c] = pd.to_numeric(anomaly_output[c], errors="coerce").fillna(0).astype(int)

print(f"\nAnomaly output: {len(anomaly_output):,} rows × {len(anomaly_output.columns)} cols")
print(f"  CRITICAL     : {(anomaly_output['anomaly_risk_level']=='CRITICAL').sum()}")
print(f"  HIGH         : {(anomaly_output['anomaly_risk_level']=='HIGH').sum()}")
print(f"  MEDIUM       : {(anomaly_output['anomaly_risk_level']=='MEDIUM').sum()}")
print(f"  LOW          : {(anomaly_output['anomaly_risk_level']=='LOW').sum()}")
print(f"  DATA_POVERTY : {(anomaly_output['anomaly_risk_level']=='DATA_POVERTY').sum()}")
print(f"  CLEAN        : {(anomaly_output['anomaly_risk_level']=='CLEAN').sum()}")
print(f"  LLM processed: {anomaly_output['llm_priority_action'].notna().sum()}")


# COMMAND ----------
# MAGIC %md ## 16. Write Anomaly Flags to Delta

# COMMAND ----------

anomaly_spark = spark.createDataFrame(
    anomaly_output.astype(object).where(anomaly_output.notna(), None)
)
(
    anomaly_spark
    .write.format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(cfg.ANOMALY_TABLE)
)
print(f"✅ Anomaly flags written → {cfg.ANOMALY_TABLE}")


# COMMAND ----------
# MAGIC %md ## 17. Region-Level Anomaly Report

# COMMAND ----------

def compute_anomaly_report(df: pd.DataFrame) -> pd.DataFrame:
    """Region-level anomaly report with IDP-native signal summary."""
    report_rows: List[Dict] = []
    stat_flag_cols  = [c for c in df.columns if c.startswith("stat_anomaly_")]
    enhanced_cols   = [c for c in df.columns if c.startswith("enhanced_")]

    for region, grp in df.groupby("region_normalised", dropna=False):
        region_str = str(region)
        if not region_str or region_str in ("nan", "None", "NaT"):
            continue

        n              = len(grp)
        flagged_n      = int((grp["total_anomaly_flags"] >= 1).sum())
        critical_n     = int((grp["anomaly_risk_level"] == "CRITICAL").sum())
        high_n         = int((grp["anomaly_risk_level"] == "HIGH").sum())
        medium_n       = int((grp["anomaly_risk_level"] == "MEDIUM").sum())
        low_n          = int((grp["anomaly_risk_level"] == "LOW").sum())
        poverty_n      = int((grp["anomaly_risk_level"] == "DATA_POVERTY").sum())
        clean_n        = int((grp["anomaly_risk_level"] == "CLEAN").sum())
        llm_confirmed  = int(grp["llm_confirmed_anomaly_count"].fillna(0).astype(int).sum())
        llm_processed  = int(grp["llm_priority_action"].notna().sum())
        avg_dq_vals    = grp["llm_data_quality_score"].dropna()
        avg_dq         = round(float(avg_dq_vals.mean()), 2) if len(avg_dq_vals) > 0 else 0.0
        avg_composite  = round(float(grp["composite_anomaly_score"].mean()), 4)
        avg_continuity = round(float(grp["continuity_risk_score"].mean()), 4)
        avg_ghost      = round(float(grp["ghost_probability_score"].mean()), 4)
        avg_qual_risk  = round(float(grp["quality_risk_score"].mean()), 4)
        avg_cap_conf   = round(float(grp["capability_confidence"].mean()), 4)
        high_cont_n    = int(grp["high_continuity_risk"].sum())
        dup_n          = int(grp["identity_duplicate_risk"].sum())
        avg_completeness  = round(float(grp["data_completeness_score"].mean()), 4)
        avg_absence_conf  = round(float(grp["evidence_absence_confidence"].mean()), 4)
        avg_evidence_wt   = round(float(grp["evidence_weight"].mean()), 4)

        mat_dist = grp["service_maturity_label"].value_counts().to_dict()

        top_flags: List[str] = []
        for fc in stat_flag_cols + enhanced_cols:
            if fc in grp.columns and grp[fc].sum() > 0:
                short = fc.replace("stat_anomaly_", "").replace("enhanced_", "+")
                top_flags.append(f"{short}:{int(grp[fc].sum())}")
        top_flags.sort(key=lambda x: -int(x.split(":")[1]))

        worst = (
            grp.nlargest(3, "composite_anomaly_score")[
                ["name", "total_anomaly_flags", "anomaly_risk_level",
                 "composite_anomaly_score", "ghost_probability_score"]
            ].to_dict("records")
        )

        report_rows.append({
            "region":                    region_str,
            "total_facilities":          n,
            "flagged_facilities":        flagged_n,
            "flag_rate":                 round(flagged_n / n, 4) if n > 0 else 0.0,
            "critical_risk":             critical_n,
            "high_risk":                 high_n,
            "medium_risk":               medium_n,
            "low_risk":                  low_n,
            "clean_facilities":          clean_n,
            "llm_processed":             llm_processed,
            "llm_confirmed_count":       llm_confirmed,
            "avg_data_quality":          avg_dq,
            "avg_composite_score":       avg_composite,
            "avg_continuity_risk":       avg_continuity,
            "high_continuity_risk_count":high_cont_n,
            "identity_duplicate_risk_count": dup_n,
            "avg_completeness":          avg_completeness,
            "avg_absence_confidence":    avg_absence_conf,
            "maturity_distribution":     json.dumps(mat_dist),
            "top_anomaly_types":         json.dumps(top_flags[:8]),
            "worst_facilities":          json.dumps(worst),
        })

    return pd.DataFrame(report_rows).sort_values("flag_rate", ascending=False).reset_index(drop=True)


report_df = compute_anomaly_report(anomaly_output)

report_spark = spark.createDataFrame(
    report_df.astype(object).where(report_df.notna(), None)
)
(
    report_spark
    .write.format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(cfg.REPORT_TABLE)
)
print(f"✅ Anomaly report written → {cfg.REPORT_TABLE}")


# COMMAND ----------
# MAGIC %md ## 18. Write Regional Priority to Delta

# COMMAND ----------

reg_priority_spark = spark.createDataFrame(
    regional_priority_df.astype(object).where(regional_priority_df.notna(), None)
)
(
    reg_priority_spark
    .write.format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(cfg.PRIORITY_TABLE)
)
print(f"✅ Regional priority written → {cfg.PRIORITY_TABLE}")


# COMMAND ----------
# MAGIC %md ## 19. MLflow Final Metrics

# COMMAND ----------

mlflow.set_experiment(cfg.MLFLOW_EXP)
with mlflow.start_run(run_name="08_anomaly_report_v4"):
    mlflow.log_metric("total_flagged",         int((anomaly_output["total_anomaly_flags"] >= 1).sum()))
    mlflow.log_metric("critical_risk_count",   int((anomaly_output["anomaly_risk_level"] == "CRITICAL").sum()))
    mlflow.log_metric("high_risk_count",       int((anomaly_output["anomaly_risk_level"] == "HIGH").sum()))
    mlflow.log_metric("data_poverty_count",    int((anomaly_output["anomaly_risk_level"] == "DATA_POVERTY").sum()))
    mlflow.log_metric("llm_confirmed_total",   int(anomaly_output["llm_confirmed_anomaly_count"].fillna(0).astype(int).sum()))
    mlflow.log_metric("regions_with_anomalies",int(report_df["flagged_facilities"].gt(0).sum()))
    mlflow.log_metric("high_continuity_total", int(anomaly_output["high_continuity_risk"].sum()))
    mlflow.log_metric("id_duplicate_total",    int(anomaly_output["identity_duplicate_risk"].sum()))
    mlflow.log_metric("avg_composite_score",   round(float(anomaly_output["composite_anomaly_score"].mean()), 4))
    mlflow.log_metric("avg_ghost_prob",        round(float(anomaly_output["ghost_probability_score"].mean()), 4))
    mlflow.log_metric("avg_evidence_weight",   round(float(anomaly_output["evidence_weight"].mean()), 4))

    for fc in STAT_FLAG_COLS + ENHANCED_COLS:
        if fc in anomaly_output.columns:
            mlflow.log_metric(f"flag_{fc}", int(anomaly_output[fc].sum()))

    for label, cnt in anomaly_output["service_maturity_label"].value_counts().items():
        mlflow.log_metric(f"maturity_{safe_str(label).lower()}", int(cnt))

    mlflow.log_dict(report_df.to_dict(orient="records"), "anomaly_report.json")
    print("✅ MLflow metrics logged.")


# COMMAND ----------
# MAGIC %md ## 20. Console Reports

# COMMAND ----------

# ── Regional report ────────────────────────────────────────────────────────────
print(f"\n{'='*82}")
print("ANOMALY REPORT BY REGION  (v4 IDP-Native)")
print(f"{'='*82}")
print(f"{'Region':<26}  {'N':>4}  {'Flag%':>6}  {'Crit':>5}  {'High':>5}  "
      f"{'LLM✓':>5}  {'ContRisk':>8}  {'AvgGhost':>8}")
print("-" * 82)
for _, r in report_df.iterrows():
    avg_ghost = anomaly_output[anomaly_output["region_normalised"] == r["region"]]["ghost_probability_score"].mean()
    print(
        f"{r['region']:<26}  {r['total_facilities']:>4}  {r['flag_rate']:>5.1%}  "
        f"{r['critical_risk']:>5}  {r['high_risk']:>5}  {r['llm_confirmed_count']:>5}  "
        f"{r['high_continuity_risk_count']:>8}  {avg_ghost:>8.3f}"
    )

# ── Top 15 highest-risk ────────────────────────────────────────────────────────
print(f"\n{'='*82}")
print("TOP 15 HIGHEST-RISK FACILITIES (by composite_anomaly_score)")
print(f"{'='*82}")
view_cols = [
    "name", "region_normalised", "facility_type_clean",
    "total_anomaly_flags", "anomaly_risk_level", "composite_anomaly_score",
    "ghost_probability_score", "quality_risk_score",
    "service_maturity_label", "evidence_absence_confidence",
    "llm_anomaly_severity", "llm_data_quality_score", "llm_priority_action",
]
disp_cols = [c for c in view_cols if c in anomaly_output.columns]
for _, r in anomaly_output.head(15)[disp_cols].iterrows():
    print(f"\n  🏥 {r.get('name','?')} [{safe_str(r.get('facility_type_clean','?')).upper()}]")
    print(f"     Region       : {r.get('region_normalised','?')}")
    print(f"     Risk Level   : {r.get('anomaly_risk_level','?')}  "
          f"(flags={r.get('total_anomaly_flags',0)}, "
          f"score={safe_float(r.get('composite_anomaly_score')):.3f})")
    print(f"     Ghost prob   : {safe_float(r.get('ghost_probability_score')):.3f} | "
          f"Quality risk: {safe_float(r.get('quality_risk_score')):.3f} | "
          f"Maturity: {r.get('service_maturity_label','?')}")
    if r.get("llm_anomaly_severity"):
        print(f"     LLM Confirm  : {r.get('llm_anomaly_severity','?')} "
              f"(DQ={safe_float(r.get('llm_data_quality_score',5.0)):.1f}/10)")
    if r.get("llm_priority_action"):
        print(f"     Action       : {str(r.get('llm_priority_action',''))[:130]}")

# ── Full evaluation summary ────────────────────────────────────────────────────
print(f"\n{'='*82}")
print("FULL EVALUATION SUMMARY  (v4 — IDP-Native)")
print(f"{'='*82}")

print(f"\n  STATISTICAL FLAGS (Layer 1c):")
for fc in STAT_FLAG_COLS:
    if fc in anomaly_output.columns:
        n = int(anomaly_output[fc].sum())
        print(f"    {fc.replace('stat_anomaly_',''):<48}  {n:>4}  ({n/len(anomaly_output):.1%})")

print(f"\n  ENHANCED FLAGS (Layers 1c–1g):")
for fc in ENHANCED_COLS + ["high_continuity_risk", "identity_duplicate_risk"]:
    if fc in anomaly_output.columns:
        n = int(anomaly_output[fc].sum())
        label = fc.replace("enhanced_", "").replace("_", " ")
        print(f"    {label:<48}  {n:>4}  ({n/len(anomaly_output):.1%})")

print(f"\n  COMPOSITE RISK LEVELS:")
for level in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "DATA_POVERTY", "CLEAN"]:
    n   = int((anomaly_output["anomaly_risk_level"] == level).sum())
    bar = "█" * max(1, n // 10)
    print(f"    {level:<16}  {n:>4}  {bar}")

print(f"\n  SERVICE MATURITY:")
for label in ["CORROBORATED", "CREDIBLE", "CLAIMED", "SPARSE", "SHELL"]:
    n = int((anomaly_output["service_maturity_label"] == label).sum())
    print(f"    {label:<16}  {n:>4}")

print(f"\n  LLM VALIDATION (Layer 2):")
print(f"    Processed           : {int(anomaly_output['llm_priority_action'].notna().sum()):>4}")
print(f"    Confirmed anomalies : {int(anomaly_output['llm_confirmed_anomaly_count'].fillna(0).astype(int).sum()):>4}")
avg_dq_val = anomaly_output["llm_data_quality_score"].dropna().mean()
print(f"    Avg DQ Score        : {avg_dq_val:.2f}/10" if avg_dq_val == avg_dq_val else "    Avg DQ Score        :  N/A")

print(f"\n  IDP-NATIVE SIGNAL QUALITY:")
print(f"    Avg evidence_weight         : {anomaly_output['evidence_weight'].mean():.3f}")
print(f"    Avg ghost_probability_score : {anomaly_output['ghost_probability_score'].mean():.3f}")
print(f"    Avg capability_confidence   : {anomaly_output['capability_confidence'].mean():.3f}")
print(f"    Avg quality_risk_score      : {anomaly_output['quality_risk_score'].mean():.3f}")
print(f"    Avg composite_anomaly_score : {anomaly_output['composite_anomaly_score'].mean():.4f}")
print(f"    Avg evidence_absence_conf   : {anomaly_output['evidence_absence_confidence'].mean():.3f}")
print(f"    High continuity risk        : {int(anomaly_output['high_continuity_risk'].sum()):>4}")
print(f"    Identity duplicate risk     : {int(anomaly_output['identity_duplicate_risk'].sum()):>4}")
print(f"    Data poverty flag           : {int(anomaly_output['data_poverty_flag'].sum()):>4}")

print(f"\n  OUTPUT TABLES:")
print(f"    {cfg.ANOMALY_TABLE}")
print(f"    {cfg.REPORT_TABLE}")
print(f"    {cfg.PRIORITY_TABLE}")
print(f"\n  Sample queries:")
print(f"    SELECT * FROM {cfg.ANOMALY_TABLE}")
print(f"      WHERE anomaly_risk_level IN ('CRITICAL','HIGH')")
print(f"      ORDER BY composite_anomaly_score DESC LIMIT 50;")
print(f"\n    SELECT * FROM {cfg.PRIORITY_TABLE}")
print(f"      ORDER BY regional_priority_score DESC;")

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_anomaly_flags

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_anomaly_report

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_anomaly_report

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_anomaly_flags